In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/medical/mtsamples.csv")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print("FOLDER:", root)
    for file in files:
        print("   FILE:", file)

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/kaggle/input/datasets/prardhanaanil/medical/mtsamples.csv"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
print(df.columns.tolist())

In [ ]:
display(df.head())

In [ ]:
print(df.isnull().sum())

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
print("Number of medical specialties:",
      df["medical_specialty"].nunique())

In [ ]:
display(df["medical_specialty"].value_counts())

In [ ]:
print(df["transcription"].iloc[0])

In [ ]:
for i in range(3):
    print("\n" + "="*100)
    print("REPORT", i+1)
    print("="*100)
    print(df["transcription"].iloc[i])

In [ ]:
df["report_length"] = df["transcription"].fillna("").str.len()

print(df["report_length"].describe())

In [ ]:
shortest = df.loc[df["report_length"].idxmin(), "transcription"]

print(shortest)

In [ ]:
shortest = df.loc[df["report_length"].idxmin(), "transcription"]

print(shortest)

In [ ]:
df = df.drop(columns=["Unnamed: 0"])

print(df.columns.tolist())

In [ ]:
print("Before:", len(df))

df = df.dropna(subset=["transcription"])

print("After:", len(df))

In [ ]:
print("Missing transcriptions:", df["transcription"].isna().sum())

In [ ]:
short_reports = df[df["transcription"].str.len() < 100]

print("Reports shorter than 100 characters:", len(short_reports))

In [ ]:
df = df[df["transcription"].str.len() >= 100]

print("Remaining reports:", len(df))

In [ ]:
import re

def clean_text(text):
    text = str(text)
    
    # Replace multiple spaces/newlines/tabs with one space
    text = re.sub(r"\s+", " ", text)
    
    # Remove spaces before punctuation
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    
    # Remove extra spaces after commas/colons
    text = re.sub(r"([,:])\s+", r"\1 ", text)
    
    return text.strip()

df["transcription"] = df["transcription"].apply(clean_text)

In [ ]:
print(df["transcription"].iloc[0])

In [ ]:
print("Duplicate transcriptions:",
      df["transcription"].duplicated().sum())

In [ ]:
df = df.drop_duplicates(
    subset=["transcription"]
).reset_index(drop=True)

print("Final number of reports:", len(df))

In [ ]:
print(df.shape)
print()
print(df.isnull().sum())

In [ ]:
df["report_length"] = df["transcription"].str.len()

print(df["report_length"].describe())

In [ ]:
clean_path = "/kaggle/working/mtsamples_clean.csv"

df.to_csv(clean_path, index=False)

print("Saved successfully!")
print(clean_path)

In [ ]:
medical_df = df.copy()

print("Working dataset shape:", medical_df.shape)

In [ ]:
specialty_counts = medical_df["medical_specialty"].value_counts()

display(specialty_counts)

In [ ]:
medical_df = medical_df.drop(columns=["report_length"])

print(medical_df.columns.tolist())

In [ ]:
medical_df["report_id"] = [
    f"REPORT_{i:04d}"
    for i in range(len(medical_df))
]

display(medical_df[[
    "report_id",
    "medical_specialty",
    "sample_name"
]].head())

In [ ]:
report_lengths = medical_df["transcription"].str.len()

print("Minimum length:", report_lengths.min())
print("Maximum length:", report_lengths.max())
print("Average length:", round(report_lengths.mean(), 2))
print("Median length:", report_lengths.median())

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_df, temp_df = train_test_split(
    medical_df,
    test_size=0.10,
    random_state=42,
    stratify=medical_df["medical_specialty"]
)

In [ ]:
specialty_counts = medical_df["medical_specialty"].value_counts()

print(specialty_counts.sort_values())

In [ ]:
split_df = medical_df.copy()

specialty_counts = split_df["medical_specialty"].value_counts()

split_df["split_specialty"] = split_df["medical_specialty"].apply(
    lambda x: x if specialty_counts[x] >= 5 else "Other"
)

print(split_df["split_specialty"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    split_df,
    test_size=0.10,
    random_state=42,
    stratify=split_df["split_specialty"]
)

In [ ]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["split_specialty"]
)

In [ ]:
# Split the 10% temporary data into validation and test
# WITHOUT stratification

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

In [ ]:
train_df = train_df.drop(columns=["split_specialty"])
val_df = val_df.drop(columns=["split_specialty"])
test_df = test_df.drop(columns=["split_specialty"])

In [ ]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

In [ ]:
train_df.to_csv(
    "/kaggle/working/train.csv",
    index=False
)

val_df.to_csv(
    "/kaggle/working/validation.csv",
    index=False
)

test_df.to_csv(
    "/kaggle/working/test.csv",
    index=False
)

print("Datasets saved successfully!")

In [ ]:
import pandas as pd
import os

DATA_DIR = "/kaggle/working"

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
val_df = pd.read_csv(f"{DATA_DIR}/validation.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

print("Training data:", train_df.shape)
print("Validation data:", val_df.shape)
print("Testing data:", test_df.shape)

print("\nTraining columns:")
print(train_df.columns.tolist())

print("\nSample report:")
print(train_df["transcription"].iloc[0][:1000])

In [ ]:
!pip install -q scispacy

In [ ]:
import sys
import spacy
import subprocess

print("Python version:", sys.version)
print("spaCy version:", spacy.__version__)

result = subprocess.run(
    ["pip", "show", "scispacy"],
    capture_output=True,
    text=True
)

print("\nSciSpaCy installation status:")
print(result.stdout if result.stdout else result.stderr)

In [ ]:
%pip install -q transformers

In [ ]:
from transformers import pipeline

model_name = "d4data/biomedical-ner-all"

ner_pipeline = pipeline(
    "token-classification",
    model=model_name,
    aggregation_strategy="simple",
    device=-1
)

print("Biomedical NER model loaded successfully!")

In [ ]:
sample_report = train_df["transcription"].iloc[0]

entities = ner_pipeline(sample_report[:2000])

print("Extracted medical entities:\n")

for entity in entities:
    print(
        "Text:", entity["word"],
        "| Type:", entity["entity_group"],
        "| Confidence:", round(entity["score"], 3)
    )

In [ ]:
import os
import urllib.request
import tarfile

# Save the report archive
download_url = (
    "https://openi.nlm.nih.gov/imgs/collections/"
    "NLMCXR_reports.tgz"
)

archive_path = "/kaggle/working/NLMCXR_reports.tgz"
extract_path = "/kaggle/working/iu_xray_reports"

os.makedirs(extract_path, exist_ok=True)

print("Downloading IU X-Ray reports...")

urllib.request.urlretrieve(download_url, archive_path)

print("Download completed!")

# Extract the report files
with tarfile.open(archive_path, "r:gz") as archive:
    archive.extractall(extract_path, filter="data")

print("Reports extracted successfully!")

print("\nExtracted folders:")
print(os.listdir(extract_path)[:10])

In [ ]:
import os

IU_REPORT_PATH = "/kaggle/working/iu_xray_reports/ecgen-radiology"

print("Checking IU X-Ray report files...\n")

all_files = []

for root, dirs, files in os.walk(IU_REPORT_PATH):
    for file in files:
        all_files.append(os.path.join(root, file))

print("Total files found:", len(all_files))

print("\nFirst 10 files:")
for file in all_files[:10]:
    print(file)

In [ ]:
import os
import xml.etree.ElementTree as ET

IU_REPORT_PATH = "/kaggle/working/iu_xray_reports/ecgen-radiology"

# Select the first XML file
xml_files = [
    f for f in os.listdir(IU_REPORT_PATH)
    if f.endswith(".xml")
]

sample_file = os.path.join(IU_REPORT_PATH, xml_files[0])

print("Sample file:", xml_files[0])

# Read the XML
tree = ET.parse(sample_file)
root = tree.getroot()

# Display the XML structure
print("\nXML report content:\n")
print(ET.tostring(root, encoding="unicode")[:5000])

In [ ]:
import os
import xml.etree.ElementTree as ET
import pandas as pd

# Path containing the 3,955 XML files
IU_REPORT_PATH = "/kaggle/working/iu_xray_reports/ecgen-radiology"

# Helper function to extract text from an XML element
def get_section_text(root, section_label):
    texts = []

    for element in root.findall(".//Abstract/AbstractText"):
        label = element.attrib.get("Label", "").strip().upper()

        if label == section_label.upper():
            text = " ".join("".join(element.itertext()).split())

            if text and text.lower() not in ["none available.", "none available"]:
                texts.append(text)

    return " ".join(texts).strip()


# Extract the IU X-Ray reports
iu_records = []

xml_files = [
    filename
    for filename in os.listdir(IU_REPORT_PATH)
    if filename.lower().endswith(".xml")
]

for filename in xml_files:
    filepath = os.path.join(IU_REPORT_PATH, filename)

    try:
        tree = ET.parse(filepath)
        root = tree.getroot()

        # Extract report identifier
        uid_element = root.find(".//uId")
        report_id = (
            uid_element.attrib.get("id")
            if uid_element is not None
            else os.path.splitext(filename)[0]
        )

        # Extract report sections
        indication = get_section_text(root, "INDICATION")
        findings = get_section_text(root, "FINDINGS")
        impression = get_section_text(root, "IMPRESSION")
        comparison = get_section_text(root, "COMPARISON")

        # Combine clinically relevant sections
        sections = []

        if indication:
            sections.append("Indication: " + indication)

        if comparison:
            sections.append("Comparison: " + comparison)

        if findings:
            sections.append("Findings: " + findings)

        if impression:
            sections.append("Impression: " + impression)

        report_text = "\n".join(sections).strip()

        # Skip records without usable report text
        if not report_text:
            continue

        iu_records.append({
            "report_id": report_id,
            "report_text": report_text,
            "indication": indication,
            "findings": findings,
            "impression": impression,
            "comparison": comparison,
            "source_dataset": "IU_XRay",
            "specialty": "Radiology"
        })

    except (ET.ParseError, OSError) as error:
        print(f"Could not process {filename}: {error}")


# Create the DataFrame
iu_df = pd.DataFrame(iu_records)

print("Extraction completed!")
print("Total XML files:", len(xml_files))
print("Reports extracted:", len(iu_df))
print("\nColumns:")
print(iu_df.columns.tolist())

print("\nFirst extracted report:")
print(iu_df["report_text"].iloc[0] if not iu_df.empty else "No reports extracted")

In [ ]:
print("Dataset shape:", iu_df.shape)

print("\nMissing values:")
print(iu_df.isnull().sum())

print("\nEmpty report sections:")
for column in ["indication", "findings", "impression", "comparison"]:
    empty_count = iu_df[column].fillna("").str.strip().eq("").sum()
    print(f"{column}: {empty_count}")

print("\nReport length statistics:")
iu_df["report_length"] = iu_df["report_text"].str.len()
print(iu_df["report_length"].describe())

print("\nSample findings:")
print(iu_df["findings"].head(3).to_string(index=False))

print("\nSample impressions:")
print(iu_df["impression"].head(3).to_string(index=False))

In [ ]:
output_path = "/kaggle/working/iu_xray_reports.csv"

iu_df.to_csv(output_path, index=False)

print("IU X-Ray dataset saved successfully!")
print("Saved to:", output_path)
print("Number of reports:", len(iu_df))

In [ ]:
import pandas as pd

# Make a copy so we preserve the original extracted data
iu_clean_df = iu_df.copy()

print("Original IU X-Ray dataset:", iu_clean_df.shape)

# 1. Remove duplicate report IDs
iu_clean_df = iu_clean_df.drop_duplicates(
    subset=["report_id"],
    keep="first"
)

print("After removing duplicate IDs:", iu_clean_df.shape)

# 2. Normalize whitespace in report text
iu_clean_df["report_text"] = (
    iu_clean_df["report_text"]
    .fillna("")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# 3. Remove empty reports
iu_clean_df = iu_clean_df[
    iu_clean_df["report_text"].str.len() > 0
].copy()

print("After removing empty reports:", iu_clean_df.shape)

# 4. Recalculate report lengths
iu_clean_df["report_length"] = iu_clean_df["report_text"].str.len()

# 5. Inspect short reports before deciding whether to remove them
print("\nReport length statistics:")
print(iu_clean_df["report_length"].describe())

print("\nReports shorter than 100 characters:",
      (iu_clean_df["report_length"] < 100).sum())

# 6. Check duplicate report text
duplicate_text_count = iu_clean_df["report_text"].duplicated().sum()

print("\nDuplicate report texts:", duplicate_text_count)

# 7. Remove exact duplicate report texts
iu_clean_df = iu_clean_df.drop_duplicates(
    subset=["report_text"],
    keep="first"
).reset_index(drop=True)

print("\nFinal cleaned IU X-Ray dataset:", iu_clean_df.shape)

print("\nSample report:")
print(iu_clean_df["report_text"].iloc[0])

# 8. Save the cleaned dataset
iu_clean_df.to_csv(
    "/kaggle/working/iu_xray_reports_clean.csv",
    index=False
)

print("\nCleaned dataset saved successfully!")

In [ ]:
import pandas as pd

# ==========================================
# 1. Prepare MTSamples
# ==========================================

mtsamples_df = df.copy()

mtsamples_df["report_text"] = (
    mtsamples_df["transcription"]
    .fillna("")
    .astype(str)
    .str.strip()
)

mtsamples_df["source_dataset"] = "MTSamples"

mtsamples_df["specialty"] = (
    mtsamples_df["medical_specialty"]
    .fillna("Unknown")
)

mtsamples_df["report_type"] = "Medical transcription"

mtsamples_df["original_report_id"] = (
    mtsamples_df.index.astype(str)
)

# Keep only the common columns
mtsamples_df = mtsamples_df[
    [
        "report_text",
        "source_dataset",
        "specialty",
        "report_type",
        "original_report_id"
    ]
].copy()


# ==========================================
# 2. Prepare IU X-Ray
# ==========================================

iu_combined_df = iu_clean_df.copy()

iu_combined_df["source_dataset"] = "IU_XRay"
iu_combined_df["specialty"] = "Radiology"
iu_combined_df["report_type"] = "Radiology report"

iu_combined_df["original_report_id"] = (
    iu_combined_df["report_id"].astype(str)
)

# Keep only the common columns
iu_combined_df = iu_combined_df[
    [
        "report_text",
        "source_dataset",
        "specialty",
        "report_type",
        "original_report_id"
    ]
].copy()


# ==========================================
# 3. Combine both datasets
# ==========================================

combined_df = pd.concat(
    [mtsamples_df, iu_combined_df],
    ignore_index=True
)

# Remove empty reports
combined_df["report_text"] = (
    combined_df["report_text"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

combined_df = combined_df[
    combined_df["report_text"].str.len() > 0
].copy()

# Create unique IDs for the combined dataset
combined_df = combined_df.reset_index(drop=True)

combined_df.insert(
    0,
    "report_id",
    [f"MED_{i:05d}" for i in range(1, len(combined_df) + 1)]
)

# Calculate report length
combined_df["report_length"] = (
    combined_df["report_text"].str.len()
)

print("Combined dataset shape:", combined_df.shape)

print("\nReports by source:")
print(combined_df["source_dataset"].value_counts())

print("\nReports by type:")
print(combined_df["report_type"].value_counts())

print("\nMissing values:")
print(combined_df.isnull().sum())

print("\nSample records:")
display(combined_df.head(3))

In [ ]:
# Normalize report text for duplicate detection
combined_df["normalized_text"] = (
    combined_df["report_text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print("Total reports:", len(combined_df))

print(
    "Duplicate normalized report texts:",
    combined_df["normalized_text"].duplicated().sum()
)

print("\nDuplicate reports by source:")
duplicate_mask = combined_df["normalized_text"].duplicated(
    keep=False
)

display(
    combined_df.loc[
        duplicate_mask,
        ["report_id", "source_dataset", "report_type", "report_text"]
    ].head(10)
)

In [ ]:
# Keep a copy before removing duplicates
combined_df = combined_df.copy()

# Count duplicate occurrences before removal
duplicate_count = combined_df["normalized_text"].duplicated().sum()

print("Duplicate occurrences found:", duplicate_count)

# Remove exact duplicates based on normalized text
combined_df = combined_df.drop_duplicates(
    subset=["normalized_text"],
    keep="first"
).copy()

# Reset index
combined_df = combined_df.reset_index(drop=True)

# Regenerate unique report IDs
combined_df["report_id"] = [
    f"MED_{i:05d}" for i in range(1, len(combined_df) + 1)
]

print("\nDataset after duplicate removal:", combined_df.shape)

print("\nReports by source:")
print(combined_df["source_dataset"].value_counts())

print("\nRemaining duplicate texts:",
      combined_df["normalized_text"].duplicated().sum())

In [ ]:
output_path = "/kaggle/working/combined_medical_reports_clean.csv"

# Remove the temporary normalization column from the saved output
combined_df.drop(
    columns=["normalized_text"],
    errors="ignore"
).to_csv(output_path, index=False)

print("Final combined dataset saved!")
print("File:", output_path)
print("Total reports:", len(combined_df))
print("File size (MB):", round(
    __import__("os").path.getsize(output_path) / (1024 * 1024), 2
))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the final combined dataset
data_path = "/kaggle/working/combined_medical_reports_clean.csv"

combined_df = pd.read_csv(data_path)

print("Dataset loaded:", combined_df.shape)

# Split 1: 80% training, 20% temporary
train_df, temp_df = train_test_split(
    combined_df,
    test_size=0.20,
    random_state=42,
    stratify=combined_df["source_dataset"]
)

# Split 2: Divide the temporary data equally
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["source_dataset"]
)

print("\nTraining set:", train_df.shape)
print("Validation set:", val_df.shape)
print("Testing set:", test_df.shape)

print("\nTraining distribution:")
print(train_df["source_dataset"].value_counts())

print("\nValidation distribution:")
print(val_df["source_dataset"].value_counts())

print("\nTesting distribution:")
print(test_df["source_dataset"].value_counts())

In [ ]:
import os

output_dir = "/kaggle/working/medical_report_splits"
os.makedirs(output_dir, exist_ok=True)

train_df.to_csv(
    f"{output_dir}/train.csv",
    index=False
)

val_df.to_csv(
    f"{output_dir}/validation.csv",
    index=False
)

test_df.to_csv(
    f"{output_dir}/test.csv",
    index=False
)

print("All splits saved successfully!")

print("\nSaved files:")
for filename in os.listdir(output_dir):
    print(filename)

In [ ]:
import torch
import pandas as pd
import os

# Check GPU availability
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

# Load our saved dataset splits
DATA_DIR = "/kaggle/working/medical_report_splits"

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
val_df = pd.read_csv(f"{DATA_DIR}/validation.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

print("\nDataset sizes")
print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

print("\nAvailable columns:")
print(train_df.columns.tolist())

print("\nSample medical report:")
print(train_df["report_text"].iloc[0][:1000])

In [ ]:
%pip install -q transformers

In [ ]:
from transformers import pipeline
import torch

model_name = "d4data/biomedical-ner-all"

device = 0 if torch.cuda.is_available() else -1

print("Loading biomedical NER model...")

ner_pipeline = pipeline(
    "token-classification",
    model=model_name,
    aggregation_strategy="simple",
    device=device
)

print("\nModel loaded successfully!")
print("Device:", "GPU" if device == 0 else "CPU")

In [ ]:
sample_report = train_df["report_text"].iloc[0]

# Keep the first test short
sample_text = sample_report[:2000]

entities = ner_pipeline(sample_text)

print("MEDICAL REPORT")
print("=" * 60)
print(sample_text[:500])

print("\nEXTRACTED ENTITIES")
print("=" * 60)

for entity in entities:
    print(
        f"Text: {entity['word']:<35} "
        f"Type: {entity['entity_group']:<20} "
        f"Confidence: {entity['score']:.3f}"
    )

In [ ]:
# Select one report from the training dataset
sample_report = train_df["report_text"].iloc[0]

# Process the first 2000 characters
sample_text = sample_report[:2000]

# Extract medical entities
entities = ner_pipeline(sample_text)

print("=" * 70)
print("SAMPLE MEDICAL REPORT")
print("=" * 70)
print(sample_text[:500])

print("\n" + "=" * 70)
print("EXTRACTED MEDICAL ENTITIES")
print("=" * 70)

for entity in entities:
    print(
        f"Text: {entity['word']:<35} "
        f"Type: {entity['entity_group']:<20} "
        f"Confidence: {entity['score']:.3f}"
    )

In [ ]:
import pandas as pd
from collections import Counter

# Select five reports from each source
sample_mts = (
    train_df[train_df["source_dataset"] == "MTSamples"]
    .head(5)
)

sample_iu = (
    train_df[train_df["source_dataset"] == "IU_XRay"]
    .head(5)
)

sample_reports = pd.concat(
    [sample_mts, sample_iu],
    ignore_index=True
)

evaluation_results = []
entity_type_counts = Counter()

print("Testing the pretrained NER model on 10 reports...\n")

for _, row in sample_reports.iterrows():

    report_text = str(row["report_text"])

    # Limit the initial test to 2000 characters per report
    text = report_text[:2000]

    entities = ner_pipeline(text)

    for entity in entities:
        evaluation_results.append({
            "report_id": row["report_id"],
            "source_dataset": row["source_dataset"],
            "entity_text": entity["word"],
            "entity_type": entity["entity_group"],
            "confidence": round(float(entity["score"]), 4)
        })

        entity_type_counts[entity["entity_group"]] += 1

print("Reports tested:", len(sample_reports))
print("Total entities extracted:", len(evaluation_results))

print("\nEntity counts by category:")
for entity_type, count in entity_type_counts.most_common():
    print(f"{entity_type}: {count}")

entities_df = pd.DataFrame(evaluation_results)

print("\nFirst 20 extracted entities:")
display(entities_df.head(20))

In [ ]:
output_path = "/kaggle/working/ner_sample_results.csv"

entities_df.to_csv(output_path, index=False)

print("NER results saved successfully!")
print("Location:", output_path)
print("Total extracted entities:", len(entities_df))

In [ ]:
import pandas as pd

# Compare the extracted entities by source
source_summary = (
    entities_df
    .groupby(["source_dataset", "entity_type"])
    .size()
    .reset_index(name="count")
    .sort_values(
        ["source_dataset", "count"],
        ascending=[True, False]
    )
)

print("Entity counts by dataset:")
display(source_summary)

print("\nTotal entities by dataset:")
display(
    entities_df.groupby("source_dataset")
    .agg(
        total_entities=("entity_text", "count"),
        unique_entities=("entity_text", "nunique"),
        average_confidence=("confidence", "mean")
    )
)

In [ ]:
print("Lowest-confidence predictions:\n")

display(
    entities_df
    .sort_values("confidence")
    [
        [
            "source_dataset",
            "entity_text",
            "entity_type",
            "confidence"
        ]
    ]
    .head(30)
)

In [ ]:
import pandas as pd

# Select 10 reports from each source
mtsamples_subset = (
    train_df[train_df["source_dataset"] == "MTSamples"]
    .head(10)
)

iu_subset = (
    train_df[train_df["source_dataset"] == "IU_XRay"]
    .head(10)
)

annotation_sample = pd.concat(
    [mtsamples_subset, iu_subset],
    ignore_index=True
)

# Extract short excerpts for manual annotation
annotation_sample["text_excerpt"] = (
    annotation_sample["report_text"]
    .str.slice(0, 700)
)

annotation_sample = annotation_sample[
    [
        "report_id",
        "source_dataset",
        "text_excerpt"
    ]
].copy()

print("Reports selected:", len(annotation_sample))

display(annotation_sample.head(10))

In [ ]:
annotation_path = "/kaggle/working/medical_ner_annotation_template.csv"

annotation_template = annotation_sample.copy()

annotation_template["verified_entities"] = ""
annotation_template["annotation_notes"] = ""

annotation_template.to_csv(
    annotation_path,
    index=False
)

print("Annotation template saved!")
print(annotation_path)

In [ ]:
import pandas as pd

annotation_path = "/kaggle/working/medical_ner_annotation_template.csv"

annotation_df = pd.read_csv(annotation_path)

# Display the complete report excerpts
pd.set_option("display.max_colwidth", None)

display(annotation_df[["report_id", "source_dataset", "text_excerpt"]])

In [ ]:
import pandas as pd
import json

annotation_path = "/kaggle/working/medical_ner_annotation_template.csv"

annotation_df = pd.read_csv(annotation_path)

starter_annotations = {
    "MED_01635": [
        {"text": "Synovitis", "label": "DISEASE"},
        {"text": "anterior cruciate ligament tear", "label": "DISEASE"},
        {"text": "left knee pain", "label": "SYMPTOM"},
        {"text": "left knee", "label": "ANATOMY"}
    ],

    "MED_01524": [
        {"text": "Left-sided weakness", "label": "SYMPTOM"},
        {"text": "fevers", "label": "SYMPTOM"},
        {"text": "chills", "label": "SYMPTOM"},
        {"text": "diaphoresis", "label": "SYMPTOM"},
        {"text": "anorexia", "label": "SYMPTOM"},
        {"text": "urinary frequency", "label": "SYMPTOM"},
        {"text": "myalgias", "label": "SYMPTOM"},
        {"text": "generalized weakness", "label": "SYMPTOM"}
    ],

    "MED_02252": [
        {"text": "chronic pain", "label": "SYMPTOM"},
        {"text": "methadone", "label": "MEDICATION"},
        {"text": "neck", "label": "ANATOMY"},
        {"text": "breast", "label": "ANATOMY"},
        {"text": "abdomen", "label": "ANATOMY"}
    ],

    "MED_01028": [
        {"text": "Dentigerous cyst", "label": "DISEASE"},
        {"text": "left mandible", "label": "ANATOMY"},
        {"text": "wisdom tooth", "label": "ANATOMY"}
    ],

    "MED_01768": [
        {"text": "FCR tendinitis", "label": "DISEASE"},
        {"text": "cortisone", "label": "MEDICATION"},
        {"text": "ibuprofen", "label": "MEDICATION"},
        {"text": "Wrist", "label": "ANATOMY"},
        {"text": "elbow", "label": "ANATOMY"},
        {"text": "shoulder", "label": "ANATOMY"},
        {"text": "neck", "label": "ANATOMY"},
        {"text": "FCR tendon", "label": "ANATOMY"},
        {"text": "scaphoid", "label": "ANATOMY"}
    ]
}

for index, row in annotation_df.iterrows():
    report_id = row["report_id"]

    if report_id in starter_annotations:
        annotation_df.at[index, "verified_entities"] = json.dumps(
            starter_annotations[report_id],
            ensure_ascii=False
        )

        annotation_df.at[index, "annotation_notes"] = (
            "Starter annotations; requires manual verification."
        )

output_path = "/kaggle/working/medical_ner_annotation_progress.csv"

annotation_df.to_csv(output_path, index=False)

print("Saved starter annotations to:", output_path)
print("Annotated reports:", len(starter_annotations))

In [ ]:
import pandas as pd

# Load the annotation progress file
annotation_path = "/kaggle/working/medical_ner_annotation_progress.csv"

annotation_df = pd.read_csv(annotation_path)

# Ensure annotation columns use the correct data type
annotation_df["verified_entities"] = (
    annotation_df["verified_entities"].fillna("").astype(str)
)

annotation_df["annotation_notes"] = (
    annotation_df["annotation_notes"].fillna("").astype(str)
)

# Save the corrected file
annotation_df.to_csv(annotation_path, index=False)

# Verify the annotations
annotated_reports = annotation_df[
    annotation_df["verified_entities"].str.strip() != ""
]

print("File saved successfully!")
print("Total reports:", len(annotation_df))
print("Reports with annotations:", len(annotated_reports))

display(
    annotated_reports[
        ["report_id", "source_dataset", "verified_entities", "annotation_notes"]
    ]
)

In [ ]:
import pandas as pd
import json

annotation_path = "/kaggle/working/medical_ner_annotation_progress.csv"

annotation_df = pd.read_csv(annotation_path)

allowed_labels = {
    "DISEASE",
    "SYMPTOM",
    "MEDICATION",
    "ANATOMY",
    "TEST"
}

issues = []

for _, row in annotation_df.iterrows():

    raw_entities = row["verified_entities"]

    if pd.isna(raw_entities) or not str(raw_entities).strip():
        continue

    try:
        entities = json.loads(raw_entities)
    except json.JSONDecodeError:
        issues.append({
            "report_id": row["report_id"],
            "entity": "",
            "issue": "Invalid JSON"
        })
        continue

    report_text = str(row["text_excerpt"])

    for entity in entities:

        entity_text = entity.get("text", "")
        entity_label = entity.get("label", "")

        if entity_label not in allowed_labels:
            issues.append({
                "report_id": row["report_id"],
                "entity": entity_text,
                "issue": f"Invalid label: {entity_label}"
            })

        if not entity_text:
            issues.append({
                "report_id": row["report_id"],
                "entity": entity_text,
                "issue": "Empty entity text"
            })

        elif entity_text.casefold() not in report_text.casefold():
            issues.append({
                "report_id": row["report_id"],
                "entity": entity_text,
                "issue": "Entity text not found in excerpt"
            })

print("Reports checked:", (annotation_df["verified_entities"].fillna("").str.strip() != "").sum())
print("Annotation issues found:", len(issues))

if issues:
    display(pd.DataFrame(issues))
else:
    print("All annotations passed these basic checks.")

In [ ]:
import pandas as pd
import json

# Load the annotated reports
annotation_path = "/kaggle/working/medical_ner_annotation_progress.csv"
annotation_df = pd.read_csv(annotation_path)

annotated_reports = annotation_df[
    annotation_df["verified_entities"].fillna("").str.strip() != ""
].copy()

# Confirm the existing NER pipeline is available
assert "ner_pipeline" in globals(), (
    "ner_pipeline is not defined. Please load your NER model first."
)

results = []

for _, row in annotated_reports.iterrows():
    report_id = row["report_id"]
    text = str(row["text_excerpt"])

    # Load manually reviewed reference annotations
    gold_entities = json.loads(row["verified_entities"])

    # Predict entities
    predictions = ner_pipeline(text)

    # Keep only our five evaluation categories
    accepted_labels = {
        "DISEASE",
        "SYMPTOM",
        "MEDICATION",
        "ANATOMY",
        "TEST"
    }

    predicted_entities = []

    for prediction in predictions:
        label = prediction.get("entity_group", prediction.get("entity", ""))

        if label in accepted_labels:
            predicted_entities.append({
                "text": prediction["word"].strip(),
                "label": label,
                "confidence": float(prediction["score"])
            })

    gold_pairs = {
        (entity["text"].casefold(), entity["label"])
        for entity in gold_entities
    }

    predicted_pairs = {
        (entity["text"].casefold(), entity["label"])
        for entity in predicted_entities
    }

    true_positives = len(gold_pairs & predicted_pairs)
    false_positives = len(predicted_pairs - gold_pairs)
    false_negatives = len(gold_pairs - predicted_pairs)

    precision = (
        true_positives / (true_positives + false_positives)
        if true_positives + false_positives else 0.0
    )

    recall = (
        true_positives / (true_positives + false_negatives)
        if true_positives + false_negatives else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    results.append({
        "report_id": report_id,
        "gold_entities": len(gold_pairs),
        "predicted_entities": len(predicted_pairs),
        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

results_df = pd.DataFrame(results)

display(results_df.round(3))

# Calculate aggregate micro metrics
tp = results_df["true_positives"].sum()
fp = results_df["false_positives"].sum()
fn = results_df["false_negatives"].sum()

micro_precision = tp / (tp + fp) if tp + fp else 0.0
micro_recall = tp / (tp + fn) if tp + fn else 0.0
micro_f1 = (
    2 * micro_precision * micro_recall / (micro_precision + micro_recall)
    if micro_precision + micro_recall else 0.0
)

print("\n--- Aggregate Results ---")
print(f"True positives: {tp}")
print(f"False positives: {fp}")
print(f"False negatives: {fn}")
print(f"Micro Precision: {micro_precision:.3f}")
print(f"Micro Recall:    {micro_recall:.3f}")
print(f"Micro F1-score:  {micro_f1:.3f}")

# Save results
results_path = "/kaggle/working/medical_ner_evaluation_results.csv"
results_df.to_csv(results_path, index=False)

print("\nResults saved to:", results_path)

In [ ]:
import pandas as pd

# Load our annotation file
annotation_path = "/kaggle/working/medical_ner_annotation_progress.csv"
annotation_df = pd.read_csv(annotation_path)

# Select the first annotated report
row = annotation_df[
    annotation_df["verified_entities"].fillna("").str.strip() != ""
].iloc[0]

text = str(row["text_excerpt"])

# Run the NER model
predictions = ner_pipeline(text)

print("REPORT ID:", row["report_id"])
print("\nFIRST 20 RAW MODEL PREDICTIONS:\n")

for prediction in predictions[:20]:
    print({
        "text": prediction["word"],
        "label": prediction.get(
            "entity_group",
            prediction.get("entity", "UNKNOWN")
        ),
        "confidence": round(float(prediction["score"]), 3)
    })

print("\nALL LABELS PRODUCED BY THE MODEL:\n")

labels = sorted({
    prediction.get(
        "entity_group",
        prediction.get("entity", "UNKNOWN")
    )
    for prediction in predictions
})

print(labels)

In [ ]:
import pandas as pd
import json

# Load annotated reports
annotation_path = "/kaggle/working/medical_ner_annotation_progress.csv"

annotation_df = pd.read_csv(annotation_path)

annotated_reports = annotation_df[
    annotation_df["verified_entities"].fillna("").str.strip() != ""
].copy()

# Map the model's labels to our evaluation labels
label_mapping = {
    "Disease_disorder": "DISEASE",
    "Sign_symptom": "SYMPTOM",
    "Medication": "MEDICATION",
    "Biological_structure": "ANATOMY",
    "Diagnostic_procedure": "TEST"
}

results = []
detailed_results = []

for _, row in annotated_reports.iterrows():

    report_id = row["report_id"]
    text = str(row["text_excerpt"])

    # Ground-truth annotations
    gold_entities = json.loads(row["verified_entities"])

    gold_pairs = {
        (entity["text"].strip().casefold(), entity["label"])
        for entity in gold_entities
    }

    # Model predictions
    predictions = ner_pipeline(text)

    predicted_pairs = set()

    for prediction in predictions:

        original_label = prediction.get(
            "entity_group",
            prediction.get("entity", "")
        )

        # Convert the model label to our label
        mapped_label = label_mapping.get(original_label)

        # Ignore categories outside our five evaluation labels
        if mapped_label is None:
            continue

        predicted_text = prediction["word"].strip()

        predicted_pairs.add(
            (predicted_text.casefold(), mapped_label)
        )

        detailed_results.append({
            "report_id": report_id,
            "predicted_text": predicted_text,
            "original_label": original_label,
            "mapped_label": mapped_label,
            "confidence": float(prediction["score"])
        })

    # Exact entity-and-label matching
    true_positives = len(gold_pairs & predicted_pairs)
    false_positives = len(predicted_pairs - gold_pairs)
    false_negatives = len(gold_pairs - predicted_pairs)

    precision = (
        true_positives / (true_positives + false_positives)
        if true_positives + false_positives > 0
        else 0.0
    )

    recall = (
        true_positives / (true_positives + false_negatives)
        if true_positives + false_negatives > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    results.append({
        "report_id": report_id,
        "gold_entities": len(gold_pairs),
        "predicted_entities": len(predicted_pairs),
        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

# Display report-level results
results_df = pd.DataFrame(results)

print("REPORT-LEVEL EVALUATION")
display(results_df.round(3))

# Aggregate micro metrics
tp = results_df["true_positives"].sum()
fp = results_df["false_positives"].sum()
fn = results_df["false_negatives"].sum()

micro_precision = tp / (tp + fp) if tp + fp else 0.0
micro_recall = tp / (tp + fn) if tp + fn else 0.0

micro_f1 = (
    2 * micro_precision * micro_recall
    / (micro_precision + micro_recall)
    if micro_precision + micro_recall
    else 0.0
)

print("\nAGGREGATE RESULTS")
print("True positives:", tp)
print("False positives:", fp)
print("False negatives:", fn)
print(f"Micro Precision: {micro_precision:.3f}")
print(f"Micro Recall:    {micro_recall:.3f}")
print(f"Micro F1-score:  {micro_f1:.3f}")

# Save results
results_path = "/kaggle/working/medical_ner_evaluation_results.csv"
details_path = "/kaggle/working/medical_ner_prediction_details.csv"

results_df.to_csv(results_path, index=False)
pd.DataFrame(detailed_results).to_csv(details_path, index=False)

print("\nSaved evaluation results:", results_path)
print("Saved prediction details:", details_path)

In [ ]:
import pandas as pd
import json

# Load the evaluation results and prediction details
results_path = "/kaggle/working/medical_ner_evaluation_results.csv"
details_path = "/kaggle/working/medical_ner_prediction_details.csv"
annotation_path = "/kaggle/working/medical_ner_annotation_progress.csv"

results_df = pd.read_csv(results_path)
details_df = pd.read_csv(details_path)
annotation_df = pd.read_csv(annotation_path)

# Select the first annotated report
report_id = "MED_01635"

row = annotation_df.loc[
    annotation_df["report_id"] == report_id
].iloc[0]

gold_entities = json.loads(row["verified_entities"])

gold_pairs = {
    (entity["text"].strip().casefold(), entity["label"])
    for entity in gold_entities
}

# Predictions for this report
report_predictions = details_df[
    details_df["report_id"] == report_id
].copy()

report_predictions["exact_match"] = report_predictions.apply(
    lambda r: (
        r["predicted_text"].strip().casefold(),
        r["mapped_label"]
    ) in gold_pairs,
    axis=1
)

print("REPORT:", report_id)

print("\nGOLD ANNOTATIONS:")
display(pd.DataFrame(gold_entities))

print("\nMODEL PREDICTIONS:")
display(
    report_predictions[
        [
            "predicted_text",
            "original_label",
            "mapped_label",
            "confidence",
            "exact_match"
        ]
    ].round(3)
)

print("\nPREDICTIONS NOT MATCHING THE GOLD ANNOTATIONS:")
display(
    report_predictions.loc[
        ~report_predictions["exact_match"],
        [
            "predicted_text",
            "original_label",
            "mapped_label",
            "confidence"
        ]
    ].round(3)
)

In [ ]:
import pandas as pd

annotation_path = "/kaggle/working/medical_ner_annotation_progress.csv"

annotation_df = pd.read_csv(annotation_path)

row = annotation_df.loc[
    annotation_df["report_id"] == "MED_01635"
].iloc[0]

text = str(row["text_excerpt"])

predictions = ner_pipeline(text)

print("REPORT ID:", row["report_id"])
print("\nPREDICTED ENTITY SPANS\n")

for prediction in predictions:
    start = prediction.get("start")
    end = prediction.get("end")

    if start is not None and end is not None:
        original_span = text[start:end]
    else:
        original_span = "Offsets unavailable"

    print({
        "predicted_text": prediction["word"],
        "original_text_span": original_span,
        "label": prediction.get(
            "entity_group",
            prediction.get("entity", "UNKNOWN")
        ),
        "start": start,
        "end": end,
        "confidence": round(float(prediction["score"]), 3)
    })

In [ ]:
import pandas as pd

annotation_path = "/kaggle/working/medical_ner_annotation_progress.csv"
annotation_df = pd.read_csv(annotation_path)

annotated_reports = annotation_df[
    annotation_df["verified_entities"].fillna("").str.strip() != ""
]

all_predictions = []

for _, row in annotated_reports.iterrows():

    text = str(row["text_excerpt"])
    predictions = ner_pipeline(text)

    for prediction in predictions:

        start = prediction.get("start")
        end = prediction.get("end")

        # Extract the exact text covered by the model's offsets
        original_span = (
            text[start:end]
            if start is not None and end is not None
            else None
        )

        predicted_text = prediction["word"].strip()

        # Check whether the prediction text matches its original span
        span_matches = (
            predicted_text.casefold() == original_span.casefold()
            if original_span is not None
            else None
        )

        all_predictions.append({
            "report_id": row["report_id"],
            "original_span": original_span,
            "predicted_text": predicted_text,
            "label": prediction.get(
                "entity_group",
                prediction.get("entity", "UNKNOWN")
            ),
            "confidence": float(prediction["score"]),
            "start": start,
            "end": end,
            "span_matches": span_matches
        })

predictions_df = pd.DataFrame(all_predictions)

print("Total predictions:", len(predictions_df))

print("\nPredictions where model text differs from original span:")
display(
    predictions_df[predictions_df["span_matches"] == False]
)

print("\nPredictions with confidence below 0.60:")
display(
    predictions_df[predictions_df["confidence"] < 0.60]
)

print("\nPrediction counts by report:")
display(
    predictions_df.groupby("report_id").size().reset_index(
        name="prediction_count"
    )
)

# Save the inspection results
output_path = "/kaggle/working/medical_ner_span_inspection.csv"
predictions_df.to_csv(output_path, index=False)

print("\nSaved inspection results to:", output_path)

In [ ]:
test_words = [
    "Synovitis",
    "anorexia",
    "myalgias",
    "ibuprofen",
    "cortisone",
    "pneumonia",
    "cardiomegaly"
]

tokenizer = ner_pipeline.tokenizer

print("Tokenizer:", tokenizer.name_or_path)
print("\nTokenization results:\n")

for word in test_words:
    tokens = tokenizer.tokenize(word)
    print(f"{word:20} -> {tokens}")

In [ ]:
import torch
from transformers import pipeline

# Select GPU if available
device = 0 if torch.cuda.is_available() else -1

print("Device:", "GPU" if device == 0 else "CPU")

# Reload the pretrained biomedical NER model
model_name = "d4data/biomedical-ner-all"

ner_pipeline = pipeline(
    "token-classification",
    model=model_name,
    aggregation_strategy="simple",
    device=device
)

print("\nNER model loaded successfully!")
print("Model:", model_name)
print("Tokenizer:", ner_pipeline.tokenizer.name_or_path)

In [ ]:
import os
from pathlib import Path

cache_dir = Path("/root/.cache/huggingface/hub")

model_folders = list(
    cache_dir.glob("models--d4data--biomedical-ner-all")
)

if not model_folders:
    print("Model cache folder not found.")
else:
    model_dir = model_folders[0]
    print("Model cache folder:", model_dir)

    safetensors_files = list(model_dir.rglob("model.safetensors"))

    if safetensors_files:
        for file in safetensors_files:
            print("Model file:", file)
            print("File size (MB):", round(file.stat().st_size / (1024**2), 2))
    else:
        print("Complete model weights were not found in the cache.")

    incomplete_files = list(model_dir.rglob("*.incomplete"))
    print("Incomplete download files:", len(incomplete_files))

In [ ]:
from huggingface_hub import snapshot_download

model_name = "d4data/biomedical-ner-all"

model_path = snapshot_download(
    repo_id=model_name,
    allow_patterns=[
        "config.json",
        "model.safetensors",
        "tokenizer.json",
        "tokenizer_config.json",
        "special_tokens_map.json",
        "vocab.txt",
        "added_tokens.json"
    ],
    max_workers=2
)

print("Model files downloaded successfully!")
print("Model location:", model_path)

In [ ]:
import torch
from transformers import pipeline

model_name = "d4data/biomedical-ner-all"

device = 0 if torch.cuda.is_available() else -1

ner_pipeline = pipeline(
    "token-classification",
    model=model_name,
    aggregation_strategy="simple",
    device=device
)

print("NER model loaded successfully!")
print("Device:", "GPU" if device == 0 else "CPU")
print("Tokenizer:", ner_pipeline.tokenizer.name_or_path)

In [ ]:
test_words = [
    "Synovitis",
    "anorexia",
    "myalgias",
    "ibuprofen",
    "cortisone",
    "pneumonia",
    "cardiomegaly"
]

tokenizer = ner_pipeline.tokenizer

print("Tokenization results:\n")

for word in test_words:
    tokens = tokenizer.tokenize(word)
    print(f"{word:20} -> {tokens}")

In [ ]:
test_texts = [
    "Diagnosis: Synovitis and anterior cruciate ligament tear of the left knee.",
    "The patient has anorexia, fever, chills, and generalized weakness.",
    "The patient takes ibuprofen and cortisone.",
    "Chest radiograph shows cardiomegaly without pneumonia."
]

for text in test_texts:
    print("\n" + "=" * 80)
    print("TEXT:", text)
    print("-" * 80)

    predictions = ner_pipeline(text)

    for prediction in predictions:
        start = prediction["start"]
        end = prediction["end"]

        print({
            "text": text[start:end],
            "label": prediction["entity_group"],
            "confidence": round(float(prediction["score"]), 3)
        })

In [ ]:
print("Pipeline aggregation strategy:")
print(ner_pipeline._preprocess_params)

print("\nTokenizer type:")
print(type(ner_pipeline.tokenizer).__name__)

print("\nModel configuration:")
print("Model type:", ner_pipeline.model.config.model_type)
print("Number of labels:", ner_pipeline.model.config.num_labels)

print("\nLabel mapping:")
print(ner_pipeline.model.config.id2label)

In [ ]:
import torch

test_text = (
    "Diagnosis: Synovitis and anterior cruciate ligament tear "
    "of the left knee. The patient takes ibuprofen and cortisone."
)

# Tokenize the input
inputs = ner_pipeline.tokenizer(
    test_text,
    return_tensors="pt",
    truncation=True
)

# Move tensors to the model's device
model_device = next(ner_pipeline.model.parameters()).device
inputs = {key: value.to(model_device) for key, value in inputs.items()}

# Get token-level predictions
ner_pipeline.model.eval()

with torch.no_grad():
    outputs = ner_pipeline.model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=-1)
predicted_ids = probabilities.argmax(dim=-1)[0]
confidence_scores = probabilities.max(dim=-1).values[0]

tokens = ner_pipeline.tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

id2label = ner_pipeline.model.config.id2label

print(f"{'TOKEN':25} {'PREDICTED LABEL':35} {'CONFIDENCE':12}")
print("-" * 75)

for token, label_id, confidence in zip(
    tokens,
    predicted_ids.tolist(),
    confidence_scores.tolist()
):
    label = id2label[label_id]

    print(f"{token:25} {label:35} {confidence:.3f}")

In [ ]:
# Inspect the current NER pipeline's post-processing configuration

print("Pipeline aggregation strategy:")
print(ner_pipeline._postprocess_params.get("aggregation_strategy"))

print("\nOther post-processing parameters:")
print(ner_pipeline._postprocess_params)

print("\nTokenizer special tokens:")
print(ner_pipeline.tokenizer.special_tokens_map)

In [ ]:
from transformers import pipeline

test_text = (
    "Diagnosis: Synovitis and anterior cruciate ligament tear "
    "of the left knee. The patient takes ibuprofen and cortisone."
)

strategies = ["simple", "first", "average", "max"]

for strategy in strategies:
    print("\n" + "=" * 70)
    print(f"AGGREGATION STRATEGY: {strategy.upper()}")
    print("=" * 70)

    test_pipeline = pipeline(
        "token-classification",
        model=ner_pipeline.model,
        tokenizer=ner_pipeline.tokenizer,
        aggregation_strategy=strategy,
        device=0 if torch.cuda.is_available() else -1
    )

    predictions = test_pipeline(test_text)

    for entity in predictions:
        print(
            f"Text: {entity['word']:<35} "
            f"Label: {entity['entity_group']:<25} "
            f"Score: {entity['score']:.3f}"
        )

In [ ]:
import pandas as pd
from transformers import pipeline

# Create a pipeline using the FIRST aggregation strategy
ner_first = pipeline(
    "token-classification",
    model=ner_pipeline.model,
    tokenizer=ner_pipeline.tokenizer,
    aggregation_strategy="first",
    device=0 if torch.cuda.is_available() else -1
)

# Select three reports from each dataset
mtsamples = combined_df[
    combined_df["source_dataset"] == "MTSamples"
].head(3)

iu_xray = combined_df[
    combined_df["source_dataset"] == "IU_XRay"
].head(3)

test_reports = pd.concat(
    [mtsamples, iu_xray],
    ignore_index=True
)

results = []

for _, row in test_reports.iterrows():
    text = str(row["report_text"])[:1500]

    # Predictions using both strategies
    simple_predictions = ner_pipeline(text)
    first_predictions = ner_first(text)

    results.append({
        "report_id": row["report_id"],
        "source_dataset": row["source_dataset"],
        "simple_entity_count": len(simple_predictions),
        "first_entity_count": len(first_predictions),
        "simple_examples": "; ".join(
            f"{e['word']} ({e['entity_group']})"
            for e in simple_predictions[:8]
        ),
        "first_examples": "; ".join(
            f"{e['word']} ({e['entity_group']})"
            for e in first_predictions[:8]
        )
    })

comparison_df = pd.DataFrame(results)

display(comparison_df)

In [ ]:
import pandas as pd
import os

dataset_path = "/kaggle/working/combined_medical_reports_clean.csv"

if os.path.exists(dataset_path):
    combined_df = pd.read_csv(dataset_path)
    print("Dataset loaded successfully!")
    print("Shape:", combined_df.shape)
    print("\nReports by dataset:")
    print(combined_df["source_dataset"].value_counts())
else:
    print("Dataset not found at:", dataset_path)

In [ ]:
import torch
import pandas as pd
from transformers import pipeline

# Create a pipeline using FIRST aggregation
ner_first = pipeline(
    "token-classification",
    model=ner_pipeline.model,
    tokenizer=ner_pipeline.tokenizer,
    aggregation_strategy="first",
    device=0 if torch.cuda.is_available() else -1
)

# Select three reports from each dataset
mtsamples = combined_df[
    combined_df["source_dataset"] == "MTSamples"
].head(3)

iu_xray = combined_df[
    combined_df["source_dataset"] == "IU_XRay"
].head(3)

test_reports = pd.concat(
    [mtsamples, iu_xray],
    ignore_index=True
)

results = []

for _, row in test_reports.iterrows():
    text = str(row["report_text"])[:1500]

    # Run both NER pipelines
    simple_predictions = ner_pipeline(text)
    first_predictions = ner_first(text)

    results.append({
        "report_id": row["report_id"],
        "source_dataset": row["source_dataset"],
        "simple_entity_count": len(simple_predictions),
        "first_entity_count": len(first_predictions),
        "simple_examples": "; ".join(
            f"{e['word']} ({e['entity_group']})"
            for e in simple_predictions[:8]
        ),
        "first_examples": "; ".join(
            f"{e['word']} ({e['entity_group']})"
            for e in first_predictions[:8]
        )
    })

comparison_df = pd.DataFrame(results)

display(comparison_df)

In [ ]:
# Select one IU X-Ray report
sample_report = combined_df[
    combined_df["source_dataset"] == "IU_XRay"
].iloc[1]

report_text = str(sample_report["report_text"])[:1500]

print("REPORT ID:", sample_report["report_id"])
print("=" * 80)
print(report_text)
print("\n")

# Generate predictions
simple_predictions = ner_pipeline(report_text)
first_predictions = ner_first(report_text)

# Display SIMPLE predictions
print("=" * 80)
print("SIMPLE AGGREGATION")
print("=" * 80)

for entity in simple_predictions:
    print(
        f"{entity['word']:<35} "
        f"{entity['entity_group']:<30} "
        f"{entity['score']:.3f}"
    )

# Display FIRST predictions
print("\n" + "=" * 80)
print("FIRST AGGREGATION")
print("=" * 80)

for entity in first_predictions:
    print(
        f"{entity['word']:<35} "
        f"{entity['entity_group']:<30} "
        f"{entity['score']:.3f}"
    )

In [ ]:
# Experiment with confidence-based filtering

confidence_threshold = 0.80

filtered_predictions = [
    entity
    for entity in first_predictions
    if entity["score"] >= confidence_threshold
]

print("Confidence threshold:", confidence_threshold)
print("Entities before filtering:", len(first_predictions))
print("Entities after filtering:", len(filtered_predictions))

print("\n" + "=" * 80)
print("ENTITIES RETAINED")
print("=" * 80)

for entity in filtered_predictions:
    print(
        f"{entity['word']:<35} "
        f"{entity['entity_group']:<30} "
        f"{entity['score']:.3f}"
    )

print("\n" + "=" * 80)
print("ENTITIES REMOVED")
print("=" * 80)

for entity in first_predictions:
    if entity["score"] < confidence_threshold:
        print(
            f"{entity['word']:<35} "
            f"{entity['entity_group']:<30} "
            f"{entity['score']:.3f}"
        )

In [ ]:
import re

def extract_medical_entities(
    text,
    confidence_threshold=0.0
):
    """
    Extract candidate medical entities from a report.

    This is an experimental NER baseline, not a diagnostic tool.
    """

    if not isinstance(text, str) or not text.strip():
        return []

    # Run the NER model using FIRST aggregation
    predictions = ner_first(text)

    entities = []

    for prediction in predictions:
        entity_text = prediction["word"].strip()
        confidence = float(prediction["score"])

        # Remove special-token remnants or empty predictions
        if not entity_text:
            continue

        # Ignore literal report placeholders such as XXXX
        if re.fullmatch(r"(?i)x{2,}", entity_text):
            continue

        # Optional experimental confidence filter
        if confidence < confidence_threshold:
            continue

        entities.append({
            "text": entity_text,
            "label": prediction["entity_group"],
            "confidence": round(confidence, 4)
        })

    return entities


# Test using our radiology report
sample_report = combined_df[
    combined_df["report_id"] == "MED_02338"
].iloc[0]

report_text = sample_report["report_text"]

entities = extract_medical_entities(
    report_text,
    confidence_threshold=0.0
)

print("Report ID:", sample_report["report_id"])
print("\nExtracted entities:")
print("-" * 75)

for entity in entities:
    print(
        f"{entity['text']:<30} "
        f"{entity['label']:<30} "
        f"{entity['confidence']:.4f}"
    )

print("\nTotal entities retained:", len(entities))

In [ ]:
import os
import pandas as pd

files_to_check = [
    "/kaggle/working/medical_ner_annotation_template.csv",
    "/kaggle/working/medical_ner_annotation_progress.csv",
    "/kaggle/working/medical_ner_evaluation_results.csv",
    "/kaggle/working/medical_ner_prediction_details.csv"
]

for path in files_to_check:
    print("\n" + "-" * 70)
    print("File:", os.path.basename(path))

    if os.path.exists(path):
        file_df = pd.read_csv(path)
        print("Status: FOUND")
        print("Shape:", file_df.shape)
        print("Columns:", file_df.columns.tolist())
        display(file_df.head(3))
    else:
        print("Status: NOT FOUND")

In [ ]:
import pandas as pd
import json

annotation_path = (
    "/kaggle/working/medical_ner_annotation_progress.csv"
)

annotation_df = pd.read_csv(annotation_path)

def parse_entities(value):
    if pd.isna(value) or not str(value).strip():
        return []

    try:
        entities = json.loads(value)
        return entities if isinstance(entities, list) else []
    except (json.JSONDecodeError, TypeError):
        return []

annotation_df["parsed_entities"] = (
    annotation_df["verified_entities"].apply(parse_entities)
)

annotation_df["entity_count"] = (
    annotation_df["parsed_entities"].apply(len)
)

annotation_df["annotation_status"] = (
    annotation_df["entity_count"]
    .apply(lambda count: "Needs annotation/review" if count == 0
           else "Starter annotations — verify manually")
)

print("Total reports:", len(annotation_df))
print("\nAnnotation status:")
print(annotation_df["annotation_status"].value_counts())

print("\nReport-level annotation summary:")

display(
    annotation_df[
        [
            "report_id",
            "source_dataset",
            "entity_count",
            "annotation_status",
            "annotation_notes"
        ]
    ]
)

In [ ]:
import pandas as pd
import json

annotation_path = (
    "/kaggle/working/medical_ner_annotation_progress.csv"
)

annotation_df = pd.read_csv(annotation_path)

# Select the five reports with existing starter annotations
review_df = annotation_df[
    annotation_df["verified_entities"].notna()
].head(5).copy()

for _, row in review_df.iterrows():

    print("\n" + "=" * 90)
    print("REPORT ID:", row["report_id"])
    print("SOURCE:", row["source_dataset"])
    print("=" * 90)

    print("\nREPORT TEXT:\n")
    print(row["text_excerpt"])

    print("\nCURRENT STARTER ANNOTATIONS:\n")

    try:
        entities = json.loads(row["verified_entities"])

        if entities:
            for entity in entities:
                print(
                    f"Text: {entity['text']:<40} "
                    f"Label: {entity['label']}"
                )
        else:
            print("No entities annotated.")

    except (json.JSONDecodeError, TypeError):
        print("Could not parse the annotations.")

    print("\nANNOTATION NOTES:")
    print(row["annotation_notes"])

In [ ]:
import os
import shutil

source_path = (
    "/kaggle/working/medical_ner_annotation_progress.csv"
)

backup_path = (
    "/kaggle/working/medical_ner_annotation_progress_backup.csv"
)

if os.path.exists(source_path):
    shutil.copy2(source_path, backup_path)

    print("Backup created successfully!")
    print("Backup location:", backup_path)
    print("Original annotations remain unchanged.")
else:
    print("Annotation file not found:", source_path)

In [ ]:
import pandas as pd
import json

annotation_path = (
    "/kaggle/working/medical_ner_annotation_progress.csv"
)

annotation_df = pd.read_csv(annotation_path)

# Our agreed annotation categories
VALID_LABELS = {
    "DISEASE",
    "SYMPTOM",
    "MEDICATION",
    "ANATOMY",
    "TEST",
    "PROCEDURE",
    "PATHOGEN"
}

valid_json_rows = 0
invalid_json_rows = []
labels_found = set()
invalid_labels = []

for index, row in annotation_df.iterrows():
    value = row["verified_entities"]

    if pd.isna(value) or not str(value).strip():
        continue

    try:
        entities = json.loads(value)
        valid_json_rows += 1

        if not isinstance(entities, list):
            invalid_json_rows.append(index)
            continue

        for entity in entities:
            label = entity.get("label")
            labels_found.add(label)

            if label not in VALID_LABELS:
                invalid_labels.append({
                    "report_id": row["report_id"],
                    "text": entity.get("text"),
                    "label": label
                })

    except (json.JSONDecodeError, TypeError):
        invalid_json_rows.append(index)

print("Total reports:", len(annotation_df))
print("Rows containing valid JSON annotations:", valid_json_rows)
print("Invalid JSON rows:", invalid_json_rows)

print("\nLabels currently present:")
print(sorted(labels_found))

print("\nLabels not in our seven-category schema:")
if invalid_labels:
    display(pd.DataFrame(invalid_labels))
else:
    print("None")

print("\nRequired categories:")
for label in sorted(VALID_LABELS):
    print("-", label)

In [ ]:
import pandas as pd
import json
import os

source_path = (
    "/kaggle/working/medical_ner_annotation_progress.csv"
)

worksheet_path = (
    "/kaggle/working/medical_ner_review_worksheet.csv"
)

df = pd.read_csv(source_path)

# Select the five reports with existing starter annotations
review_df = df[
    df["verified_entities"].notna()
].copy()

# Preserve the original annotations and add review fields
review_df["reviewed_entities"] = ""
review_df["review_status"] = "PENDING"
review_df["review_notes"] = ""

# Show the existing annotations in a readable format
review_df["starter_entity_summary"] = review_df[
    "verified_entities"
].apply(
    lambda value: "; ".join(
        f"{entity['text']} [{entity['label']}]"
        for entity in json.loads(value)
    )
)

# Save a separate worksheet
review_df.to_csv(worksheet_path, index=False)

print("Review worksheet created successfully!")
print("Location:", worksheet_path)
print("Reports included:", len(review_df))

display(
    review_df[
        [
            "report_id",
            "source_dataset",
            "starter_entity_summary",
            "review_status",
            "review_notes"
        ]
    ]
)

In [ ]:
import pandas as pd

annotation_path = (
    "/kaggle/working/medical_ner_annotation_progress.csv"
)

annotation_df = pd.read_csv(annotation_path)

# Select reports that do not yet have annotations
unannotated_df = annotation_df[
    annotation_df["verified_entities"].isna()
    | annotation_df["verified_entities"].astype(str).str.strip().eq("")
].copy()

print("Unannotated reports:", len(unannotated_df))

for _, row in unannotated_df.iterrows():
    print("\n" + "=" * 80)
    print("REPORT ID:", row["report_id"])
    print("SOURCE:", row["source_dataset"])
    print("-" * 80)
    print(row["text_excerpt"])

In [ ]:
import pandas as pd
import json
import os

source_path = (
    "/kaggle/working/medical_ner_annotation_progress.csv"
)

proposed_path = (
    "/kaggle/working/medical_ner_proposed_annotations.csv"
)

# Read the current annotation file
annotation_df = pd.read_csv(source_path)

# Store proposals separately so they cannot overwrite existing annotations
proposals = {
    "MED_00063": [
        {"text": "Penile skin bridges", "label": "DISEASE"},
        {"text": "Excision of penile skin bridges", "label": "PROCEDURE"},
        {"text": "Penis", "label": "ANATOMY"}
    ],
    "MED_00866": [
        {"text": "Blood loss anemia", "label": "DISEASE"},
        {"text": "Diverticulosis coli", "label": "DISEASE"},
        {"text": "Internal hemorrhoids", "label": "DISEASE"},
        {"text": "Colonoscopy", "label": "TEST"},
        {"text": "EGD", "label": "TEST"},
        {"text": "Flexible sigmoidoscopy", "label": "TEST"}
    ],
    "MED_02321": [
        {"text": "Syncope", "label": "SYMPTOM"},
        {"text": "Holter monitor", "label": "TEST"},
        {"text": "Bradycardia", "label": "DISEASE"},
        {"text": "Premature ventricular contraction", "label": "DISEASE"},
        {"text": "Atrial premature contractions", "label": "DISEASE"}
    ],
    "MED_00907": [
        {"text": "Right eye", "label": "ANATOMY"},
        {"text": "Cornea", "label": "ANATOMY"},
        {"text": "Anterior chamber", "label": "ANATOMY"},
        {"text": "Capsulorrhexis", "label": "PROCEDURE"}
    ],
    "MED_02280": [
        {"text": "Cystic fibrosis", "label": "DISEASE"},
        {"text": "Chronic cough", "label": "SYMPTOM"},
        {"text": "Albuterol", "label": "MEDICATION"}
    ],
    "MED_03739": [
        {"text": "Mediastinal contour", "label": "ANATOMY"},
        {"text": "Lungs", "label": "ANATOMY"}
    ],
    "MED_04039": [
        {"text": "Left arm pain", "label": "SYMPTOM"},
        {"text": "Nausea", "label": "SYMPTOM"},
        {"text": "Cardiomegaly", "label": "DISEASE"}
    ],
    "MED_04978": [
        {"text": "Chest pain", "label": "SYMPTOM"},
        {"text": "Shortness of breath", "label": "SYMPTOM"},
        {"text": "Left pleural effusion", "label": "DISEASE"},
        {"text": "Airspace disease", "label": "DISEASE"},
        {"text": "Cardiomegaly", "label": "DISEASE"}
    ],
    "MED_03835": [
        {"text": "Shortness of breath", "label": "SYMPTOM"},
        {"text": "Wheezing", "label": "SYMPTOM"},
        {"text": "Lung", "label": "ANATOMY"},
        {"text": "Pleural effusion", "label": "DISEASE"}
    ],
    "MED_02419": [
        {"text": "Pneumothorax", "label": "DISEASE"},
        {"text": "Airspace consolidation", "label": "DISEASE"},
        {"text": "Pleural effusion", "label": "DISEASE"}
    ],
    "MED_04146": [
        {"text": "Edema", "label": "SYMPTOM"},
        {"text": "Cardiomegaly", "label": "DISEASE"},
        {"text": "Pulmonary edema", "label": "DISEASE"},
        {"text": "Pleural effusion", "label": "DISEASE"}
    ],
    "MED_05863": [
        {"text": "Slurred speech", "label": "SYMPTOM"},
        {"text": "Confusion", "label": "SYMPTOM"},
        {"text": "Pleural effusion", "label": "DISEASE"},
        {"text": "Pneumonia", "label": "DISEASE"},
        {"text": "Thoracic vertebral body", "label": "ANATOMY"}
    ],
    "MED_03722": [
        {"text": "Chest pain", "label": "SYMPTOM"},
        {"text": "Dyspnea", "label": "SYMPTOM"},
        {"text": "Pleural effusion", "label": "DISEASE"}
    ],
    "MED_05464": [
        {"text": "Peripheral vascular disease", "label": "DISEASE"},
        {"text": "Chest X-ray", "label": "TEST"}
    ],
    "MED_02408": [
        {"text": "Chest pain", "label": "SYMPTOM"},
        {"text": "Shortness of breath", "label": "SYMPTOM"},
        {"text": "Chest radiographs", "label": "TEST"},
        {"text": "Atelectasis", "label": "DISEASE"},
        {"text": "Pleural effusion", "label": "DISEASE"},
        {"text": "Pneumothorax", "label": "DISEASE"},
        {"text": "Lymph nodes", "label": "ANATOMY"},
        {"text": "Cervical spine", "label": "ANATOMY"}
    ]
}

# Keep the proposals separate from verified annotations
proposal_rows = []

for report_id, entities in proposals.items():
    proposal_rows.append({
        "report_id": report_id,
        "proposed_entities": json.dumps(entities),
        "review_status": "PENDING_MANUAL_VERIFICATION",
        "review_notes": ""
    })

proposed_df = pd.DataFrame(proposal_rows)
proposed_df.to_csv(proposed_path, index=False)

print("Proposed annotation file saved:")
print(proposed_path)
print("\nReports with proposed annotations:", len(proposed_df))
print("\nAll proposals remain unverified.")
display(proposed_df[["report_id", "review_status"]])

In [ ]:
import pandas as pd
import json
import re

# Load the source reports and proposed annotations
annotation_path = (
    "/kaggle/working/medical_ner_annotation_progress.csv"
)
proposal_path = (
    "/kaggle/working/medical_ner_proposed_annotations.csv"
)

source_df = pd.read_csv(annotation_path)
proposal_df = pd.read_csv(proposal_path)

# Seven agreed categories
VALID_LABELS = {
    "DISEASE",
    "SYMPTOM",
    "MEDICATION",
    "ANATOMY",
    "TEST",
    "PROCEDURE",
    "PATHOGEN"
}

# Map each report ID to its actual text excerpt
report_texts = dict(
    zip(source_df["report_id"], source_df["text_excerpt"])
)

validation_results = []

for _, row in proposal_df.iterrows():
    report_id = row["report_id"]
    text = report_texts.get(report_id, "")

    entities = json.loads(row["proposed_entities"])

    for entity in entities:
        entity_text = entity["text"]
        label = entity["label"]

        # Check for an exact, case-insensitive phrase match.
        # Word boundaries help prevent matching partial words.
        pattern = (
            r"(?<!\w)"
            + re.escape(entity_text)
            + r"(?!\w)"
        )

        present = bool(
            re.search(pattern, str(text), flags=re.IGNORECASE)
        )

        validation_results.append({
            "report_id": report_id,
            "entity": entity_text,
            "label": label,
            "appears_in_text": present,
            "valid_label": label in VALID_LABELS
        })

validation_df = pd.DataFrame(validation_results)

print("Total proposed entities:", len(validation_df))
print(
    "Entities found verbatim:",
    int(validation_df["appears_in_text"].sum())
)
print(
    "Entities not found verbatim:",
    int((~validation_df["appears_in_text"]).sum())
)
print(
    "Entities with valid labels:",
    int(validation_df["valid_label"].sum())
)

print("\nEntities requiring attention:")
display(
    validation_df[
        (~validation_df["appears_in_text"])
        | (~validation_df["valid_label"])
    ]
)

validation_path = (
    "/kaggle/working/medical_ner_proposal_validation.csv"
)
validation_df.to_csv(validation_path, index=False)

print("\nValidation report saved to:", validation_path)

In [ ]:
import pandas as pd
import json

proposal_path = (
    "/kaggle/working/medical_ner_proposed_annotations.csv"
)

corrected_path = (
    "/kaggle/working/medical_ner_proposed_annotations_v2.csv"
)

proposal_df = pd.read_csv(proposal_path)

# Apply corrections to the affected reports
for index, row in proposal_df.iterrows():

    entities = json.loads(row["proposed_entities"])
    report_id = row["report_id"]

    if report_id == "MED_00063":
        # "Penis" is not present verbatim.
        entities = [
            e for e in entities
            if e["text"] != "Penis"
        ]

    elif report_id == "MED_02280":
        # The excerpt is truncated; do not assume the medication.
        entities = [
            e for e in entities
            if e["text"] != "Albuterol"
        ]

    elif report_id == "MED_05464":
        # The X-ray phrase contains missing placeholders.
        entities = [
            e for e in entities
            if e["text"] != "Chest X-ray"
        ]

    elif report_id == "MED_02408":
        updated_entities = []

        for entity in entities:
            if entity["text"] == "Chest radiographs":
                updated_entities.append({
                    "text": "radiographs of the chest",
                    "label": "TEST"
                })

            elif entity["text"] != "Lymph nodes":
                updated_entities.append(entity)

        entities = updated_entities

    proposal_df.at[index, "proposed_entities"] = json.dumps(
        entities,
        ensure_ascii=False
    )

    proposal_df.at[index, "review_status"] = (
        "PENDING_MANUAL_VERIFICATION"
    )

proposal_df.to_csv(corrected_path, index=False)

print("Corrected proposal file saved:")
print(corrected_path)

print("\nReports:", len(proposal_df))
print(
    "Total proposed entities:",
    sum(
        len(json.loads(value))
        for value in proposal_df["proposed_entities"]
    )
)

print("\nAll corrected proposals remain unverified.")

In [ ]:
import json
import os

# Final annotation categories
NER_LABELS = {
    "DISEASE": "Diseases and disorders",
    "SYMPTOM": "Symptoms and clinical findings",
    "MEDICATION": "Medication names",
    "ANATOMY": "Body parts and anatomical structures",
    "TEST": "Diagnostic tests and investigations",
    "PROCEDURE": "Surgical and therapeutic procedures",
    "PATHOGEN": "Infectious organisms",
    "RADIOLOGICAL_FINDING": "Findings described by medical imaging"
}

schema = {
    "schema_version": "1.0",
    "project": "Medical Report Analyzer",
    "labels": NER_LABELS,
    "annotation_rules": [
        "Annotate only information supported by the report text.",
        "Use complete entity spans consistently.",
        "Do not infer a diagnosis from a symptom alone.",
        "Do not reconstruct text hidden by XXXX placeholders.",
        "Distinguish radiological findings from confirmed diseases.",
        "Keep uncertain annotations pending manual review."
    ]
}

schema_path = "/kaggle/working/medical_ner_annotation_schema.json"

with open(schema_path, "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=4, ensure_ascii=False)

print("NER annotation schema created successfully!")
print("Location:", schema_path)
print("\nCategories:")

for label, description in NER_LABELS.items():
    print(f"{label}: {description}")

print("\nTotal categories:", len(NER_LABELS))

In [ ]:
import pandas as pd
import json
import os

# Load the annotation schema
schema_path = "/kaggle/working/medical_ner_annotation_schema.json"

with open(schema_path, "r", encoding="utf-8") as f:
    schema = json.load(f)

valid_labels = set(schema["labels"].keys())

# Load the latest proposed annotations
annotations_path = (
    "/kaggle/working/medical_ner_proposed_annotations_v2.csv"
)

annotations_df = pd.read_csv(annotations_path)

print("Total proposed annotations:", len(annotations_df))
print("Unique reports:", annotations_df["report_id"].nunique())
print("\nColumns:")
print(annotations_df.columns.tolist())

# Identify the column containing entity labels
label_column = next(
    (
        col for col in annotations_df.columns
        if col.strip().lower() in ["label", "entity_label", "category"]
    ),
    None
)

if label_column is None:
    print("\nCould not automatically identify the label column.")
    print("Please share the column names printed above.")
else:
    print("\nLabel distribution:")
    print(annotations_df[label_column].value_counts(dropna=False))

    invalid_labels = annotations_df[
        ~annotations_df[label_column].isin(valid_labels)
    ]

    print("\nAnnotations using labels outside the schema:",
          len(invalid_labels))

    if len(invalid_labels) > 0:
        display(invalid_labels)
    else:
        print("All proposed labels match the 8-category schema.")

# Confirm that these are proposals, not verified gold annotations
if "verification_status" in annotations_df.columns:
    print("\nVerification status:")
    print(
        annotations_df["verification_status"]
        .value_counts(dropna=False)
    )
else:
    print(
        "\nReminder: These proposals must be manually reviewed "
        "before being used as gold annotations."
    )

In [ ]:
import pandas as pd
import json

annotations_path = (
    "/kaggle/working/medical_ner_proposed_annotations_v2.csv"
)

annotations_df = pd.read_csv(annotations_path)

# Inspect the first three reports
for i in range(min(3, len(annotations_df))):
    row = annotations_df.iloc[i]

    print("=" * 70)
    print("Report ID:", row["report_id"])
    print("Review status:", row["review_status"])
    print("\nProposed entities:")

    entities = row["proposed_entities"]

    # Display the stored structure
    print(entities)
    print("\nPython data type:", type(entities).__name__)
    print()

In [ ]:
import pandas as pd
import json

# Load the schema
schema_path = "/kaggle/working/medical_ner_annotation_schema.json"

with open(schema_path, "r", encoding="utf-8") as f:
    schema = json.load(f)

valid_labels = set(schema["labels"].keys())

# Load proposed annotations
annotations_path = (
    "/kaggle/working/medical_ner_proposed_annotations_v2.csv"
)

annotations_df = pd.read_csv(annotations_path)

validation_results = []

for _, row in annotations_df.iterrows():

    report_id = row["report_id"]

    # Parse the JSON string into a Python list
    try:
        entities = json.loads(row["proposed_entities"])
    except (json.JSONDecodeError, TypeError):
        validation_results.append({
            "report_id": report_id,
            "entity_text": None,
            "label": None,
            "schema_valid": False,
            "issue": "Invalid JSON"
        })
        continue

    for entity in entities:

        entity_text = entity.get("text", "")
        label = entity.get("label", "")

        if label not in valid_labels:
            issue = "Label not in schema"
        elif not entity_text.strip():
            issue = "Empty entity text"
        else:
            issue = "None"

        validation_results.append({
            "report_id": report_id,
            "entity_text": entity_text,
            "label": label,
            "schema_valid": issue == "None",
            "issue": issue
        })

validation_df = pd.DataFrame(validation_results)

# Display summary
print("REPORTS CHECKED:", annotations_df["report_id"].nunique())
print("TOTAL PROPOSED ENTITIES:", len(validation_df))
print("SCHEMA-VALID ENTITIES:", validation_df["schema_valid"].sum())
print("ENTITIES WITH FORMAT/LABEL ISSUES:",
      (~validation_df["schema_valid"]).sum())

print("\nENTITY LABEL COUNTS:")
print(validation_df["label"].value_counts(dropna=False))

print("\nVALIDATION ISSUES:")
issues_df = validation_df[~validation_df["schema_valid"]]

if issues_df.empty:
    print("No schema or formatting issues found.")
else:
    display(issues_df)

# Save validation results separately
output_path = "/kaggle/working/medical_ner_schema_validation.csv"
validation_df.to_csv(output_path, index=False)

print("\nValidation results saved to:", output_path)

In [ ]:
import pandas as pd
import json

# Load proposed annotations
input_path = (
    "/kaggle/working/medical_ner_proposed_annotations_v2.csv"
)

annotations_df = pd.read_csv(input_path)

review_rows = []

for _, row in annotations_df.iterrows():

    entities = json.loads(row["proposed_entities"])

    for entity in entities:
        review_rows.append({
            "report_id": row["report_id"],
            "entity_text": entity["text"],
            "proposed_label": entity["label"],
            "text_present_in_source": "",
            "label_correct": "",
            "final_label": "",
            "review_notes": "",
            "verification_status": "PENDING"
        })

review_df = pd.DataFrame(review_rows)

output_path = (
    "/kaggle/working/medical_ner_manual_review.csv"
)

review_df.to_csv(output_path, index=False)

print("Manual review sheet created!")
print("Total entities to review:", len(review_df))
print("Saved to:", output_path)

print("\nPreview:")
display(review_df.head(10))

In [ ]:
import pandas as pd
import json

# Load the original combined dataset
data_path = (
    "/kaggle/working/combined_medical_reports_clean.csv"
)

combined_df = pd.read_csv(data_path)

# Load proposed annotations
annotations_path = (
    "/kaggle/working/medical_ner_proposed_annotations_v2.csv"
)

annotations_df = pd.read_csv(annotations_path)

# Select the first three reports
review_report_ids = annotations_df["report_id"].head(3).tolist()

for report_id in review_report_ids:

    report = combined_df[
        combined_df["report_id"] == report_id
    ]

    print("\n" + "=" * 90)

    if report.empty:
        print("Report not found:", report_id)
        continue

    report = report.iloc[0]

    print("REPORT ID:", report_id)
    print("SOURCE:", report["source_dataset"])
    print("REPORT TYPE:", report["report_type"])

    print("\nCOMPLETE REPORT TEXT:\n")
    print(report["report_text"])

    proposal_row = annotations_df[
        annotations_df["report_id"] == report_id
    ].iloc[0]

    entities = json.loads(proposal_row["proposed_entities"])

    print("\nPROPOSED ENTITIES TO REVIEW:")

    for entity in entities:
        print(
            f"- {entity['text']} | "
            f"Proposed label: {entity['label']}"
        )

print("\n" + "=" * 90)
print("Review these reports against their original text.")

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")

# Record provisional review decisions for the first three reports.
# These are not yet marked as fully verified.

decisions = {
    ("MED_00063", "Penile skin bridges"): {
        "text_present_in_source": "YES",
        "label_correct": "REVIEW",
        "final_label": "DISEASE",
        "review_notes": (
            "Exact text is supported by the preoperative and postoperative "
            "diagnoses. Confirm the schema convention for anatomical conditions."
        ),
    },
    ("MED_00063", "Excision of penile skin bridges"): {
        "text_present_in_source": "YES",
        "label_correct": "YES",
        "final_label": "PROCEDURE",
        "review_notes": "Explicitly listed as the procedure performed.",
    },
    ("MED_00866", "Blood loss anemia"): {
        "text_present_in_source": "YES",
        "label_correct": "YES",
        "final_label": "DISEASE",
        "review_notes": "Explicitly documented as the preoperative diagnosis.",
    },
    ("MED_00866", "Diverticulosis coli"): {
        "text_present_in_source": "YES",
        "label_correct": "YES",
        "final_label": "DISEASE",
        "review_notes": "Explicitly documented as a postoperative diagnosis.",
    },
    ("MED_00866", "Internal hemorrhoids"): {
        "text_present_in_source": "YES",
        "label_correct": "YES",
        "final_label": "DISEASE",
        "review_notes": "Explicitly documented as a postoperative diagnosis.",
    },
    ("MED_00866", "Colonoscopy"): {
        "text_present_in_source": "YES",
        "label_correct": "REVIEW",
        "final_label": "TEST",
        "review_notes": (
            "Diagnostic endoscopic investigation; apply the project-wide "
            "TEST versus PROCEDURE convention consistently."
        ),
    },
    ("MED_00866", "EGD"): {
        "text_present_in_source": "YES",
        "label_correct": "YES",
        "final_label": "TEST",
        "review_notes": "Named diagnostic endoscopic investigation.",
    },
    ("MED_00866", "Flexible sigmoidoscopy"): {
        "text_present_in_source": "YES",
        "label_correct": "REVIEW",
        "final_label": "TEST",
        "review_notes": (
            "Diagnostic endoscopic investigation; apply the project-wide "
            "TEST versus PROCEDURE convention consistently."
        ),
    },
    ("MED_02321", "Syncope"): {
        "text_present_in_source": "YES",
        "label_correct": "YES",
        "final_label": "SYMPTOM",
        "review_notes": "Explicitly listed as the indication for monitoring.",
    },
    ("MED_02321", "Holter monitor"): {
        "text_present_in_source": "YES",
        "label_correct": "YES",
        "final_label": "TEST",
        "review_notes": "Identifies the cardiac monitoring investigation.",
    },
    ("MED_02321", "Bradycardia"): {
        "text_present_in_source": "YES",
        "label_correct": "REVIEW",
        "final_label": "SYMPTOM",
        "review_notes": (
            "Documented slow-heart-rate finding. Review whether SYMPTOM "
            "or another clinical-finding category best fits the schema."
        ),
    },
    ("MED_02321", "Premature ventricular contraction"): {
        "text_present_in_source": "YES",
        "label_correct": "REVIEW",
        "final_label": "DISEASE",
        "review_notes": (
            "Cardiac rhythm finding. Confirm the intended category for "
            "documented arrhythmias; singular wording differs from the report's "
            "plural wording."
        ),
    },
    ("MED_02321", "Atrial premature contractions"): {
        "text_present_in_source": "YES",
        "label_correct": "REVIEW",
        "final_label": "DISEASE",
        "review_notes": (
            "Cardiac rhythm finding. Confirm the intended category for "
            "documented arrhythmias."
        ),
    },
}

for (report_id, entity_text), decision in decisions.items():
    mask = (
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    )

    for column, value in decision.items():
        review_df.loc[mask, column] = value

    # These decisions remain provisional until final confirmation.
    review_df.loc[mask, "verification_status"] = "PENDING"

review_df.to_csv(review_path, index=False)

# Show the updated first three reports
display(
    review_df[
        review_df["report_id"].isin(
            ["MED_00063", "MED_00866", "MED_02321"]
        )
    ][
        [
            "report_id",
            "entity_text",
            "proposed_label",
            "text_present_in_source",
            "label_correct",
            "final_label",
            "verification_status",
            "review_notes",
        ]
    ]
)

print("\nUpdated review sheet:", review_path)
print("All 13 entities remain pending final confirmation.")

In [ ]:
import json

schema_path = (
    "/kaggle/working/medical_ner_annotation_schema.json"
)

# Load the existing schema
with open(schema_path, "r", encoding="utf-8") as f:
    schema = json.load(f)

# Add the new category
schema["labels"]["CLINICAL_FINDING"] = (
    "Objective clinical findings and physiological abnormalities"
)

# Add a rule clarifying the distinction
new_rule = (
    "Distinguish symptoms reported by patients from objective "
    "clinical findings and physiological abnormalities."
)

if new_rule not in schema["annotation_rules"]:
    schema["annotation_rules"].append(new_rule)

# Save the updated schema
with open(schema_path, "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=4, ensure_ascii=False)

print("Updated NER annotation schema successfully!")
print("Total categories:", len(schema["labels"]))

print("\nCategories:")
for label, description in schema["labels"].items():
    print(f"{label}: {description}")

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")

# Map cardiac findings to the new category
cardiac_entities = {
    "Bradycardia": (
        "Slow-heart-rate finding documented in the Holter report."
    ),
    "Premature ventricular contraction": (
        "Cardiac rhythm finding. The source uses the plural "
        "'premature ventricular contraction'; review exact span "
        "consistency before final verification."
    ),
    "Atrial premature contractions": (
        "Cardiac rhythm finding documented in the Holter report."
    )
}

for entity_text, note in cardiac_entities.items():

    mask = (
        (review_df["report_id"] == "MED_02321")
        & (review_df["entity_text"] == entity_text)
    )

    if mask.any():
        review_df.loc[mask, "final_label"] = "CLINICAL_FINDING"
        review_df.loc[mask, "label_correct"] = "REVIEW"
        review_df.loc[mask, "review_notes"] = note
        review_df.loc[mask, "verification_status"] = "PENDING"

# Save changes
review_df.to_csv(review_path, index=False)

# Display the updated entries
display(
    review_df[
        (review_df["report_id"] == "MED_02321")
    ][
        [
            "report_id",
            "entity_text",
            "proposed_label",
            "final_label",
            "label_correct",
            "verification_status",
            "review_notes"
        ]
    ]
)

print("\nUpdated review sheet:", review_path)
print("Cardiac annotations remain pending final verification.")

In [ ]:
import pandas as pd
import json

# Load original reports
data_path = "/kaggle/working/combined_medical_reports_clean.csv"
combined_df = pd.read_csv(data_path)

# Load proposed annotations
annotations_path = (
    "/kaggle/working/medical_ner_proposed_annotations_v2.csv"
)
annotations_df = pd.read_csv(annotations_path)

# Review reports 4-6 in the proposal sheet
review_report_ids = annotations_df["report_id"].iloc[3:6].tolist()

for report_id in review_report_ids:

    report_rows = combined_df[
        combined_df["report_id"] == report_id
    ]

    print("\n" + "=" * 90)

    if report_rows.empty:
        print("Report not found:", report_id)
        continue

    report = report_rows.iloc[0]

    print("REPORT ID:", report_id)
    print("SOURCE:", report["source_dataset"])
    print("REPORT TYPE:", report["report_type"])

    print("\nCOMPLETE REPORT TEXT:\n")
    print(report["report_text"])

    proposal_row = annotations_df[
        annotations_df["report_id"] == report_id
    ].iloc[0]

    entities = json.loads(proposal_row["proposed_entities"])

    print("\nPROPOSED ENTITIES TO REVIEW:")

    for entity in entities:
        print(
            f"- {entity['text']} | "
            f"Proposed label: {entity['label']}"
        )

print("\n" + "=" * 90)
print("End of reports 4-6.")

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")

decisions = {
    ("MED_00907", "Right eye"): (
        "ANATOMY",
        "Anatomical location explicitly mentioned in the source."
    ),
    ("MED_00907", "Cornea"): (
        "ANATOMY",
        "Anatomical structure explicitly mentioned in the source."
    ),
    ("MED_00907", "Anterior chamber"): (
        "ANATOMY",
        "Anatomical structure explicitly mentioned in the source."
    ),
    ("MED_00907", "Capsulorrhexis"): (
        "PROCEDURE",
        "Surgical procedure explicitly described in the operative report."
    ),
    ("MED_02280", "Cystic fibrosis"): (
        "DISEASE",
        "Explicitly documented as a diagnosis and listed in the impression."
    ),
    ("MED_02280", "Chronic cough"): (
        "SYMPTOM",
        "Explicitly described in the patient's clinical history."
    ),
    ("MED_03739", "Mediastinal contour"): (
        "ANATOMY",
        "Anatomical structure mentioned in the radiology report."
    ),
    ("MED_03739", "Lungs"): (
        "ANATOMY",
        "Anatomical structure mentioned in the radiology report."
    ),
}

for (report_id, entity_text), (label, note) in decisions.items():

    mask = (
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    )

    if not mask.any():
        print(f"WARNING: Entity not found: {report_id} | {entity_text}")
        continue

    review_df.loc[mask, "text_present_in_source"] = "YES"
    review_df.loc[mask, "label_correct"] = "YES"
    review_df.loc[mask, "final_label"] = label
    review_df.loc[mask, "review_notes"] = note
    review_df.loc[mask, "verification_status"] = "PENDING"

review_df.to_csv(review_path, index=False)

# Display the updated entries
updated_ids = ["MED_00907", "MED_02280", "MED_03739"]

display(
    review_df[review_df["report_id"].isin(updated_ids)][
        [
            "report_id",
            "entity_text",
            "proposed_label",
            "text_present_in_source",
            "label_correct",
            "final_label",
            "verification_status",
            "review_notes",
        ]
    ]
)

print("\nUpdated review sheet:", review_path)
print("Eight provisional decisions recorded.")
print("Verification status remains PENDING.")

In [ ]:
import pandas as pd
import json

# Load original reports
data_path = "/kaggle/working/combined_medical_reports_clean.csv"
combined_df = pd.read_csv(data_path)

# Load proposed annotations
annotations_path = (
    "/kaggle/working/medical_ner_proposed_annotations_v2.csv"
)
annotations_df = pd.read_csv(annotations_path)

# Select reports 7-9 from the proposal list
review_report_ids = annotations_df["report_id"].iloc[6:9].tolist()

for report_id in review_report_ids:

    report_rows = combined_df[
        combined_df["report_id"] == report_id
    ]

    print("\n" + "=" * 90)

    if report_rows.empty:
        print("Report not found:", report_id)
        continue

    report = report_rows.iloc[0]

    print("REPORT ID:", report_id)
    print("SOURCE:", report["source_dataset"])
    print("REPORT TYPE:", report["report_type"])

    print("\nCOMPLETE REPORT TEXT:\n")
    print(report["report_text"])

    proposal_row = annotations_df[
        annotations_df["report_id"] == report_id
    ].iloc[0]

    entities = json.loads(proposal_row["proposed_entities"])

    print("\nPROPOSED ENTITIES TO REVIEW:")

    for entity in entities:
        print(
            f"- {entity['text']} | "
            f"Proposed label: {entity['label']}"
        )

print("\n" + "=" * 90)
print("End of reports 7-9.")

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")

decisions = {
    # MED_04039
    ("MED_04039", "Left arm pain"): (
        "SYMPTOM",
        "Explicitly stated in the report indication."
    ),
    ("MED_04039", "Nausea"): (
        "SYMPTOM",
        "Explicitly stated in the report indication."
    ),
    ("MED_04039", "Cardiomegaly"): (
        "RADIOLOGICAL_FINDING",
        "Imaging finding described in the radiology impression."
    ),

    # MED_04978
    ("MED_04978", "Chest pain"): (
        "SYMPTOM",
        "Explicitly stated in the report indication."
    ),
    ("MED_04978", "Shortness of breath"): (
        "SYMPTOM",
        "Explicitly stated in the report indication."
    ),
    ("MED_04978", "Left pleural effusion"): (
        "RADIOLOGICAL_FINDING",
        "Imaging finding. Preserve the visible entity without reconstructing XXXX."
    ),
    ("MED_04978", "Airspace disease"): (
        "RADIOLOGICAL_FINDING",
        "Finding described in the radiology impression."
    ),
    ("MED_04978", "Cardiomegaly"): (
        "RADIOLOGICAL_FINDING",
        "Imaging finding described in the radiology impression."
    ),

    # MED_03835
    ("MED_03835", "Shortness of breath"): (
        "SYMPTOM",
        "Explicitly stated in the report indication."
    ),
    ("MED_03835", "Wheezing"): (
        "SYMPTOM",
        "Explicitly stated in the report indication."
    ),
    ("MED_03835", "Lung"): (
        "ANATOMY",
        "Anatomical term appearing in the radiology report."
    ),
    ("MED_03835", "Pleural effusion"): (
        "RADIOLOGICAL_FINDING",
        "The entity occurs in a negated statement: 'No pleural effusion'."
    ),
}

for (report_id, entity_text), (label, note) in decisions.items():

    mask = (
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    )

    if not mask.any():
        print(
            f"WARNING: Entity not found: {report_id} | {entity_text}"
        )
        continue

    review_df.loc[mask, "text_present_in_source"] = "YES"
    review_df.loc[mask, "final_label"] = label
    review_df.loc[mask, "label_correct"] = "REVIEW"
    review_df.loc[mask, "review_notes"] = note
    review_df.loc[mask, "verification_status"] = "PENDING"

review_df.to_csv(review_path, index=False)

updated_ids = ["MED_04039", "MED_04978", "MED_03835"]

display(
    review_df[review_df["report_id"].isin(updated_ids)][
        [
            "report_id",
            "entity_text",
            "proposed_label",
            "text_present_in_source",
            "label_correct",
            "final_label",
            "verification_status",
            "review_notes",
        ]
    ]
)

print("\nUpdated review sheet:", review_path)
print("All 12 annotations remain pending final verification.")

In [ ]:
import pandas as pd
import json

# Load the original reports
combined_df = pd.read_csv(
    "/kaggle/working/combined_medical_reports_clean.csv"
)

# Load the proposed annotations
annotations_df = pd.read_csv(
    "/kaggle/working/medical_ner_proposed_annotations_v2.csv"
)

# Select reports 10-12 from the proposal list
review_report_ids = annotations_df["report_id"].iloc[9:12].tolist()

for report_id in review_report_ids:

    report_rows = combined_df[
        combined_df["report_id"] == report_id
    ]

    print("\n" + "=" * 90)

    if report_rows.empty:
        print("Report not found:", report_id)
        continue

    report = report_rows.iloc[0]

    print("REPORT ID:", report_id)
    print("SOURCE:", report["source_dataset"])
    print("REPORT TYPE:", report["report_type"])

    print("\nCOMPLETE REPORT TEXT:\n")
    print(report["report_text"])

    proposal_row = annotations_df[
        annotations_df["report_id"] == report_id
    ].iloc[0]

    entities = json.loads(proposal_row["proposed_entities"])

    print("\nPROPOSED ENTITIES TO REVIEW:")

    for entity in entities:
        print(
            f"- {entity['text']} | "
            f"Proposed label: {entity['label']}"
        )

print("\n" + "=" * 90)
print("End of reports 10-12.")

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")

decisions = {
    # MED_02419
    ("MED_02419", "Pneumothorax"): (
        "RADIOLOGICAL_FINDING",
        "Mentioned in 'No pneumothorax'; finding is explicitly negated."
    ),
    ("MED_02419", "Airspace consolidation"): (
        "RADIOLOGICAL_FINDING",
        "Mentioned in a negated radiological statement."
    ),
    ("MED_02419", "Pleural effusion"): (
        "RADIOLOGICAL_FINDING",
        "Mentioned in 'No ... pleural effusion'; finding is negated."
    ),

    # MED_04146
    ("MED_04146", "Edema"): (
        "SYMPTOM",
        "Listed as the indication. Keep pending because the precise meaning "
        "of the indication should be distinguished from the imaging findings."
    ),
    ("MED_04146", "Cardiomegaly"): (
        "RADIOLOGICAL_FINDING",
        "Stable cardiomegaly is described in the radiology findings."
    ),
    ("MED_04146", "Pulmonary edema"): (
        "RADIOLOGICAL_FINDING",
        "The report states there is no overt pulmonary edema."
    ),
    ("MED_04146", "Pleural effusion"): (
        "RADIOLOGICAL_FINDING",
        "The report states there is no large pleural effusion; preserve the "
        "qualification 'large'."
    ),

    # MED_05863
    ("MED_05863", "Slurred speech"): (
        "SYMPTOM",
        "Mentioned in the report indication; surrounding source text is "
        "partially corrupted."
    ),
    ("MED_05863", "Confusion"): (
        "SYMPTOM",
        "Mentioned in the report indication; surrounding source text is "
        "partially corrupted."
    ),
    ("MED_05863", "Pleural effusion"): (
        "RADIOLOGICAL_FINDING",
        "Explicitly negated: 'There is no pleural effusion'."
    ),
    ("MED_05863", "Pneumonia"): (
        "RADIOLOGICAL_FINDING",
        "The report says no focal air-space opacity suggests pneumonia."
    ),
    ("MED_05863", "Thoracic vertebral body"): (
        "ANATOMY",
        "Anatomical structure explicitly mentioned in the report."
    ),
}

for (report_id, entity_text), (label, note) in decisions.items():

    mask = (
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    )

    if not mask.any():
        print(f"WARNING: Entity not found: {report_id} | {entity_text}")
        continue

    review_df.loc[mask, "text_present_in_source"] = "YES"
    review_df.loc[mask, "final_label"] = label
    review_df.loc[mask, "review_notes"] = note
    review_df.loc[mask, "verification_status"] = "PENDING"

    # Keep the ambiguous indication unresolved.
    if report_id == "MED_04146" and entity_text == "Edema":
        review_df.loc[mask, "label_correct"] = "REVIEW"
    else:
        review_df.loc[mask, "label_correct"] = "REVIEW"

review_df.to_csv(review_path, index=False)

updated_ids = ["MED_02419", "MED_04146", "MED_05863"]

display(
    review_df[review_df["report_id"].isin(updated_ids)][
        [
            "report_id",
            "entity_text",
            "proposed_label",
            "text_present_in_source",
            "label_correct",
            "final_label",
            "verification_status",
            "review_notes",
        ]
    ]
)

print("\nUpdated review sheet:", review_path)
print("All decisions remain pending final verification.")

In [ ]:
import pandas as pd
import json

# Load original reports
combined_df = pd.read_csv(
    "/kaggle/working/combined_medical_reports_clean.csv"
)

# Load proposed annotations
annotations_df = pd.read_csv(
    "/kaggle/working/medical_ner_proposed_annotations_v2.csv"
)

# Select the final three reports
review_report_ids = annotations_df["report_id"].iloc[12:15].tolist()

for report_id in review_report_ids:

    report_rows = combined_df[
        combined_df["report_id"] == report_id
    ]

    print("\n" + "=" * 90)

    if report_rows.empty:
        print("Report not found:", report_id)
        continue

    report = report_rows.iloc[0]

    print("REPORT ID:", report_id)
    print("SOURCE:", report["source_dataset"])
    print("REPORT TYPE:", report["report_type"])

    print("\nCOMPLETE REPORT TEXT:\n")
    print(report["report_text"])

    proposal_row = annotations_df[
        annotations_df["report_id"] == report_id
    ].iloc[0]

    entities = json.loads(proposal_row["proposed_entities"])

    print("\nPROPOSED ENTITIES TO REVIEW:")

    for entity in entities:
        print(
            f"- {entity['text']} | "
            f"Proposed label: {entity['label']}"
        )

print("\n" + "=" * 90)
print("End of reports 13-15.")

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")

decisions = {
    # MED_03722
    ("MED_03722", "Chest pain"): (
        "SYMPTOM",
        "Explicitly listed in the report indication."
    ),
    ("MED_03722", "Dyspnea"): (
        "SYMPTOM",
        "Explicitly listed in the report indication."
    ),
    ("MED_03722", "Pleural effusion"): (
        "RADIOLOGICAL_FINDING",
        "Explicitly negated by 'There is no pleural effusion'."
    ),

    # MED_05464
    ("MED_05464", "Peripheral vascular disease"): (
        "DISEASE",
        "Explicitly documented in the indication for preoperative evaluation."
    ),

    # MED_02408
    ("MED_02408", "Chest pain"): (
        "SYMPTOM",
        "Explicitly mentioned in the report indication."
    ),
    ("MED_02408", "Shortness of breath"): (
        "SYMPTOM",
        "Explicitly mentioned in the report indication."
    ),
    ("MED_02408", "radiographs of the chest"): (
        "TEST",
        "Diagnostic imaging investigation described in the report."
    ),
    ("MED_02408", "Atelectasis"): (
        "RADIOLOGICAL_FINDING",
        "Visible entity in a partially masked sentence; review assertion."
    ),
    ("MED_02408", "Pleural effusion"): (
        "RADIOLOGICAL_FINDING",
        "Explicitly negated in the report."
    ),
    ("MED_02408", "Pneumothorax"): (
        "RADIOLOGICAL_FINDING",
        "Explicitly negated in the report."
    ),
    ("MED_02408", "Cervical spine"): (
        "ANATOMY",
        "Anatomical structure mentioned in a partially masked sentence."
    ),
}

for (report_id, entity_text), (label, note) in decisions.items():

    mask = (
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    )

    if not mask.any():
        print(f"WARNING: Entity not found: {report_id} | {entity_text}")
        continue

    review_df.loc[mask, "text_present_in_source"] = "YES"
    review_df.loc[mask, "label_correct"] = "REVIEW"
    review_df.loc[mask, "final_label"] = label
    review_df.loc[mask, "review_notes"] = note
    review_df.loc[mask, "verification_status"] = "PENDING"

review_df.to_csv(review_path, index=False)

updated_ids = ["MED_03722", "MED_05464", "MED_02408"]

display(
    review_df[review_df["report_id"].isin(updated_ids)][
        [
            "report_id",
            "entity_text",
            "proposed_label",
            "text_present_in_source",
            "label_correct",
            "final_label",
            "verification_status",
            "review_notes",
        ]
    ]
)

print("\nUpdated review sheet:", review_path)
print("Final three reports recorded.")
print("All annotations remain pending final verification.")

In [ ]:
import pandas as pd
import json

# File paths
review_path = "/kaggle/working/medical_ner_manual_review.csv"
schema_path = "/kaggle/working/medical_ner_annotation_schema.json"

# Load data
review_df = pd.read_csv(review_path, dtype=str).fillna("")

with open(schema_path, "r", encoding="utf-8") as f:
    schema = json.load(f)

valid_labels = set(schema["labels"].keys())

print("=" * 65)
print("MEDICAL NER ANNOTATION AUDIT")
print("=" * 65)

# 1. Basic counts
print("\n1. BASIC COUNTS")
print("Total annotation rows:", len(review_df))
print("Unique reports:", review_df["report_id"].nunique())
print("Unique entity texts:", review_df["entity_text"].nunique())

# 2. Report coverage
print("\n2. REPORT COVERAGE")
print("Reports represented:")
print(sorted(review_df["report_id"].unique()))

# 3. Revised label validity
print("\n3. REVISED LABEL VALIDITY")

invalid_labels = review_df[
    ~review_df["final_label"].isin(valid_labels)
]

print("Valid schema categories:", len(valid_labels))
print("Rows with invalid or missing final labels:", len(invalid_labels))

if not invalid_labels.empty:
    display(
        invalid_labels[
            ["report_id", "entity_text", "final_label"]
        ]
    )

# 4. Verification status
print("\n4. VERIFICATION STATUS")
print(review_df["verification_status"].value_counts(dropna=False))

# 5. Duplicate report/entity pairs
print("\n5. DUPLICATE CHECK")

duplicate_mask = review_df.duplicated(
    subset=["report_id", "entity_text"],
    keep=False
)

duplicates = review_df[duplicate_mask]

print("Duplicate report/entity rows:", len(duplicates))

if not duplicates.empty:
    display(
        duplicates[
            ["report_id", "entity_text", "proposed_label", "final_label"]
        ]
    )

# 6. Category changes
print("\n6. PROPOSED LABEL VS REVISED LABEL")

label_changes = review_df[
    review_df["proposed_label"] != review_df["final_label"]
]

print("Rows with changed labels:", len(label_changes))

if not label_changes.empty:
    display(
        label_changes[
            [
                "report_id",
                "entity_text",
                "proposed_label",
                "final_label",
                "verification_status",
            ]
        ]
    )

# 7. Save audit summary
summary = {
    "total_annotation_rows": int(len(review_df)),
    "unique_reports": int(review_df["report_id"].nunique()),
    "invalid_final_labels": int(len(invalid_labels)),
    "duplicate_report_entity_rows": int(len(duplicates)),
    "changed_label_rows": int(len(label_changes)),
    "verification_status_counts": {
        str(k): int(v)
        for k, v in review_df["verification_status"]
        .value_counts(dropna=False).items()
    },
}

summary_path = "/kaggle/working/medical_ner_annotation_audit.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

print("\nAudit summary saved to:", summary_path)
print("\nAudit completed. No annotations were automatically verified.")

In [ ]:
import json

assertion_schema = {
    "schema_version": "1.0",
    "project": "Medical Report Analyzer",
    "assertion_statuses": {
        "PRESENT": (
            "The report explicitly describes the entity or finding as present."
        ),
        "ABSENT": (
            "The report explicitly negates the entity or finding."
        ),
        "UNCERTAIN": (
            "The report expresses uncertainty or describes a possible finding."
        ),
        "HISTORICAL": (
            "The report describes a past condition or finding."
        ),
        "NOT_SPECIFIED": (
            "The entity is mentioned, but its assertion status is not clear."
        )
    },
    "annotation_rules": [
        "Record entity category separately from assertion status.",
        "Do not interpret a mentioned entity as automatically present.",
        "Preserve explicit negation from the source report.",
        "Preserve uncertainty expressed in the source report.",
        "Do not infer assertion status from the entity category alone.",
        "Do not reconstruct text masked by XXXX."
    ]
}

output_path = (
    "/kaggle/working/medical_ner_assertion_schema.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(assertion_schema, f, indent=4, ensure_ascii=False)

print("Assertion schema created successfully!")
print("Location:", output_path)

print("\nAvailable assertion statuses:")
for label, description in assertion_schema["assertion_statuses"].items():
    print(f"{label}: {description}")

print("\nTotal assertion statuses:",
      len(assertion_schema["assertion_statuses"]))

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")

# Add assertion columns if they do not already exist.
if "assertion_status" not in review_df.columns:
    review_df["assertion_status"] = "NOT_SPECIFIED"

if "assertion_notes" not in review_df.columns:
    review_df["assertion_notes"] = ""

# Explicitly negated radiological findings.
absent_findings = {
    ("MED_02419", "Pneumothorax"),
    ("MED_02419", "Airspace consolidation"),
    ("MED_02419", "Pleural effusion"),
    ("MED_03835", "Pleural effusion"),
    ("MED_04146", "Pulmonary edema"),
    ("MED_04146", "Pleural effusion"),
    ("MED_05863", "Pleural effusion"),
    ("MED_05863", "Pneumonia"),
    ("MED_03722", "Pleural effusion"),
    ("MED_02408", "Pleural effusion"),
    ("MED_02408", "Pneumothorax"),
}

for report_id, entity_text in absent_findings:
    mask = (
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    )

    review_df.loc[mask, "assertion_status"] = "ABSENT"
    review_df.loc[mask, "assertion_notes"] = (
        "The report explicitly negates this finding; verify the full context."
    )

# Findings explicitly described as present.
present_findings = {
    ("MED_04039", "Cardiomegaly"),
    ("MED_04978", "Left pleural effusion"),
    ("MED_04978", "Airspace disease"),
    ("MED_04978", "Cardiomegaly"),
    ("MED_04146", "Cardiomegaly"),
    ("MED_05863", "Thoracic vertebral body"),
    ("MED_05464", "Peripheral vascular disease"),
}

for report_id, entity_text in present_findings:
    mask = (
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    )

    review_df.loc[mask, "assertion_status"] = "PRESENT"
    review_df.loc[mask, "assertion_notes"] = (
        "The report documents the entity or finding; verify the full context."
    )

# Explicitly uncertain imaging finding.
uncertain_findings = {
    ("MED_03835", "Lung"),
}

# Do not mark the whole entity "Lung" as uncertain merely because
# a nearby nodule is uncertain. Keep its assertion unspecified.
# The uncertainty will be recorded in notes for manual follow-up.
mask = (
    (review_df["report_id"] == "MED_03835")
    & (review_df["entity_text"] == "Lung")
)

review_df.loc[mask, "assertion_notes"] = (
    "The nearby nodular opacity is described as possibly representing "
    "an infiltrate or nodule. This uncertainty does not apply to the "
    "anatomical entity 'Lung' itself."
)

# The remaining entities stay NOT_SPECIFIED until individually reviewed.
# No row is marked as verified by this cell.
review_df["verification_status"] = "PENDING"

review_df.to_csv(review_path, index=False)

print("Assertion columns added successfully!")
print("Total annotation rows:", len(review_df))

print("\nAssertion status counts:")
print(review_df["assertion_status"].value_counts(dropna=False))

print("\nPreview of findings with assertion status:")
display(
    review_df[
        review_df["assertion_status"] != "NOT_SPECIFIED"
    ][
        [
            "report_id",
            "entity_text",
            "final_label",
            "assertion_status",
            "assertion_notes",
            "verification_status",
        ]
    ]
)

print("\nUpdated review sheet:", review_path)
print("All annotations remain pending final verification.")

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")

mask = (
    (review_df["report_id"] == "MED_04146")
    & (review_df["entity_text"] == "Pleural effusion")
)

if mask.any():
    review_df.loc[mask, "assertion_status"] = "NOT_SPECIFIED"
    review_df.loc[mask, "assertion_notes"] = (
        "The report states 'No large pleural effusion'. "
        "This is a qualified negative finding; preserve the qualifier "
        "'large' rather than treating it as an unqualified absence."
    )
    review_df.loc[mask, "verification_status"] = "PENDING"

review_df.to_csv(review_path, index=False)

print("Qualified-negation review updated.")

display(
    review_df[
        (review_df["report_id"] == "MED_04146")
        & (review_df["entity_text"] == "Pleural effusion")
    ][
        [
            "report_id",
            "entity_text",
            "final_label",
            "assertion_status",
            "assertion_notes",
            "verification_status",
        ]
    ]
)

print("\nUpdated file:", review_path)

In [ ]:
import pandas as pd
import json

review_path = "/kaggle/working/medical_ner_manual_review.csv"
schema_path = "/kaggle/working/medical_ner_assertion_schema.json"

review_df = pd.read_csv(review_path, dtype=str).fillna("")

with open(schema_path, "r", encoding="utf-8") as f:
    assertion_schema = json.load(f)

valid_assertions = set(
    assertion_schema["assertion_statuses"].keys()
)

print("=" * 60)
print("ASSERTION ANNOTATION AUDIT")
print("=" * 60)

# 1. Check row and report counts
print("\nTotal annotation rows:", len(review_df))
print("Unique reports:", review_df["report_id"].nunique())

# 2. Check assertion statuses
invalid_assertions = review_df[
    ~review_df["assertion_status"].isin(valid_assertions)
]

print("\nInvalid or missing assertion statuses:", len(invalid_assertions))

if not invalid_assertions.empty:
    display(
        invalid_assertions[
            ["report_id", "entity_text", "assertion_status"]
        ]
    )

# 3. Assertion distribution
print("\nAssertion status distribution:")
print(review_df["assertion_status"].value_counts(dropna=False))

# 4. Verification status distribution
print("\nVerification status distribution:")
print(review_df["verification_status"].value_counts(dropna=False))

# 5. Missing required fields
required_columns = [
    "report_id",
    "entity_text",
    "final_label",
    "assertion_status",
    "verification_status",
]

missing_counts = (
    review_df[required_columns]
    .replace("", pd.NA)
    .isna()
    .sum()
)

print("\nMissing required fields:")
print(missing_counts)

# 6. Show all rows still requiring verification
pending_df = review_df[
    review_df["verification_status"] != "VERIFIED"
]

print("\nRows requiring verification:", len(pending_df))

print("\nAudit complete.")

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
reports_path = "/kaggle/working/combined_medical_reports_clean.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")
reports_df = pd.read_csv(reports_path, dtype=str).fillna("")

# Select the first report in the review sheet
report_id = review_df.iloc[0]["report_id"]

# Retrieve its original report text
report_match = reports_df[reports_df["report_id"] == report_id]

if report_match.empty:
    print("Report not found:", report_id)
else:
    report_text = report_match.iloc[0]["report_text"]

    print("=" * 70)
    print("REPORT FOR MANUAL ANNOTATION REVIEW")
    print("=" * 70)
    print("Report ID:", report_id)
    print("Dataset:", report_match.iloc[0]["source_dataset"])

    print("\nORIGINAL REPORT TEXT:\n")
    print(report_text)

    print("\nPROPOSED ANNOTATIONS:\n")
    display(
        review_df[review_df["report_id"] == report_id][
            [
                "entity_text",
                "proposed_label",
                "final_label",
                "assertion_status",
                "label_correct",
                "verification_status",
                "review_notes",
                "assertion_notes",
            ]
        ]
    )

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")

# Reviewed entities for MED_00063
updates = {
    "Penile skin bridges": {
        "label_correct": "YES",
        "final_label": "DISEASE",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Confirmed in the preoperative and postoperative diagnoses. "
            "DISEASE is the chosen category under the current annotation schema."
        ),
        "assertion_notes": (
            "Explicitly documented as a diagnosis; not negated or uncertain."
        ),
    },
    "Excision of penile skin bridges": {
        "label_correct": "YES",
        "final_label": "PROCEDURE",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Explicitly documented as the procedure performed."
        ),
        "assertion_notes": (
            "The report states that the procedure was performed."
        ),
    },
}

for entity_text, values in updates.items():
    mask = (
        (review_df["report_id"] == "MED_00063")
        & (review_df["entity_text"] == entity_text)
    )

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one matching row for {entity_text!r}; "
            f"found {int(mask.sum())}."
        )

    for column, value in values.items():
        review_df.loc[mask, column] = value

    # Mark reviewed only after accepting the source and label.
    review_df.loc[mask, "verification_status"] = "VERIFIED"

review_df.to_csv(review_path, index=False)

print("MED_00063 review saved.")
display(
    review_df[review_df["report_id"] == "MED_00063"][
        [
            "report_id",
            "entity_text",
            "final_label",
            "assertion_status",
            "label_correct",
            "verification_status",
        ]
    ]
)

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
reports_path = "/kaggle/working/combined_medical_reports_clean.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")
reports_df = pd.read_csv(reports_path, dtype=str).fillna("")

# Find the next report that has not been reviewed
pending_ids = review_df.loc[
    review_df["verification_status"] != "VERIFIED",
    "report_id"
].drop_duplicates()

if pending_ids.empty:
    print("All reports in the review sheet have been reviewed.")
else:
    report_id = pending_ids.iloc[0]

    report_match = reports_df[
        reports_df["report_id"] == report_id
    ]

    if report_match.empty:
        print("Original report not found:", report_id)
    else:
        print("=" * 70)
        print("REPORT FOR MANUAL ANNOTATION REVIEW")
        print("=" * 70)
        print("Report ID:", report_id)
        print("Dataset:", report_match.iloc[0]["source_dataset"])

        print("\nORIGINAL REPORT TEXT:\n")
        print(report_match.iloc[0]["report_text"])

        print("\nPROPOSED ANNOTATIONS:\n")
        display(
            review_df[review_df["report_id"] == report_id][
                [
                    "entity_text",
                    "proposed_label",
                    "final_label",
                    "assertion_status",
                    "label_correct",
                    "verification_status",
                    "review_notes",
                    "assertion_notes",
                ]
            ]
        )

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")

updates = {
    "Blood loss anemia": {
        "final_label": "DISEASE",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Explicitly documented as the preoperative diagnosis "
            "and described in the medical history."
        ),
        "assertion_notes": (
            "Documented as a diagnosis; not negated or uncertain."
        ),
    },
    "Diverticulosis coli": {
        "final_label": "DISEASE",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Explicitly listed as a postoperative diagnosis."
        ),
        "assertion_notes": (
            "Documented as a postoperative diagnosis."
        ),
    },
    "Internal hemorrhoids": {
        "final_label": "DISEASE",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Explicitly listed as a postoperative diagnosis "
            "and observed during the examination."
        ),
        "assertion_notes": (
            "The report documents internal hemorrhoids as present."
        ),
    },
    "Colonoscopy": {
        "final_label": "TEST",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "A diagnostic endoscopic investigation explicitly "
            "documented as performed."
        ),
        "assertion_notes": (
            "The colonoscopy was performed during this encounter."
        ),
    },
    "EGD": {
        "final_label": "TEST",
        "label_correct": "YES",
        "assertion_status": "HISTORICAL",
        "review_notes": (
            "A named diagnostic endoscopic investigation "
            "documented as previously performed."
        ),
        "assertion_notes": (
            "The report states that the patient underwent EGD "
            "during an earlier evaluation."
        ),
    },
    "Flexible sigmoidoscopy": {
        "final_label": "TEST",
        "label_correct": "YES",
        "assertion_status": "HISTORICAL",
        "review_notes": (
            "A diagnostic endoscopic investigation documented "
            "as performed during an earlier attempt."
        ),
        "assertion_notes": (
            "The report describes a prior flexible sigmoidoscopy "
            "following poor bowel preparation."
        ),
    },
}

report_id = "MED_00866"

# Validate all entities before making changes.
for entity_text in updates:
    matches = review_df[
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one row for {entity_text!r}; "
            f"found {len(matches)}."
        )

# Apply the reviewed annotations.
for entity_text, values in updates.items():
    mask = (
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    )

    for column, value in values.items():
        review_df.loc[mask, column] = value

    review_df.loc[mask, "verification_status"] = "VERIFIED"

review_df.to_csv(review_path, index=False)

print("MED_00866 annotations saved successfully.")
display(
    review_df[review_df["report_id"] == report_id][
        [
            "report_id",
            "entity_text",
            "final_label",
            "assertion_status",
            "label_correct",
            "verification_status",
        ]
    ]
)

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
review_df = pd.read_csv(review_path, dtype=str).fillna("")

verified = (review_df["verification_status"] == "VERIFIED").sum()
pending = (review_df["verification_status"] != "VERIFIED").sum()

print("=" * 50)
print("ANNOTATION REVIEW PROGRESS")
print("=" * 50)
print(f"Total entities: {len(review_df)}")
print(f"Verified entities: {verified}")
print(f"Pending entities: {pending}")
print(f"Completion: {verified / len(review_df) * 100:.1f}%")

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
reports_path = "/kaggle/working/combined_medical_reports_clean.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")
reports_df = pd.read_csv(reports_path, dtype=str).fillna("")

pending_ids = (
    review_df.loc[
        review_df["verification_status"] != "VERIFIED",
        "report_id"
    ]
    .drop_duplicates()
)

if pending_ids.empty:
    print("All proposed annotations have been reviewed.")
else:
    report_id = pending_ids.iloc[0]
    report = reports_df[reports_df["report_id"] == report_id]

    if report.empty:
        print("Original report not found:", report_id)
    else:
        print("=" * 70)
        print("REPORT FOR MANUAL ANNOTATION REVIEW")
        print("=" * 70)
        print("Report ID:", report_id)
        print("Dataset:", report.iloc[0]["source_dataset"])

        print("\nORIGINAL REPORT TEXT:\n")
        print(report.iloc[0]["report_text"])

        print("\nPROPOSED ANNOTATIONS:\n")
        display(
            review_df[review_df["report_id"] == report_id][
                [
                    "entity_text",
                    "proposed_label",
                    "final_label",
                    "assertion_status",
                    "label_correct",
                    "verification_status",
                    "review_notes",
                    "assertion_notes",
                ]
            ]
        )

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
review_df = pd.read_csv(review_path, dtype=str).fillna("")

report_id = "MED_02321"

updates = {
    "Syncope": {
        "final_label": "SYMPTOM",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Explicitly listed as the indication for the Holter "
            "monitoring investigation."
        ),
        "assertion_notes": (
            "Syncope is documented as the indication for investigation."
        ),
    },
    "Holter monitor": {
        "final_label": "TEST",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Identifies the cardiac monitoring investigation "
            "documented in the report."
        ),
        "assertion_notes": (
            "The report documents the Holter monitoring investigation."
        ),
    },
    "Bradycardia": {
        "final_label": "CLINICAL_FINDING",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Objective cardiac rate finding documented in the report."
        ),
        "assertion_notes": (
            "Bradycardia is explicitly documented in the Holter summary."
        ),
    },
    "Premature ventricular contraction": {
        "final_label": "CLINICAL_FINDING",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Cardiac rhythm finding explicitly documented in the report."
        ),
        "assertion_notes": (
            "The report documents premature ventricular contractions."
        ),
    },
    "Atrial premature contractions": {
        "final_label": "CLINICAL_FINDING",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Cardiac rhythm finding explicitly documented in the report."
        ),
        "assertion_notes": (
            "The report documents atrial premature contractions."
        ),
    },
}

# Validate all rows before updating the file.
for entity_text in updates:
    matches = review_df[
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected one row for {entity_text!r}; found {len(matches)}."
        )

# Apply the reviewed decisions.
for entity_text, values in updates.items():
    mask = (
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    )

    for column, value in values.items():
        review_df.loc[mask, column] = value

    review_df.loc[mask, "verification_status"] = "VERIFIED"

review_df.to_csv(review_path, index=False)

print(f"{report_id} annotations saved successfully.")

display(
    review_df[review_df["report_id"] == report_id][
        [
            "entity_text",
            "final_label",
            "assertion_status",
            "label_correct",
            "verification_status",
        ]
    ]
)

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
review_df = pd.read_csv(review_path, dtype=str).fillna("")

total = len(review_df)
verified = (review_df["verification_status"] == "VERIFIED").sum()
pending = total - verified

print("=" * 45)
print("ANNOTATION REVIEW PROGRESS")
print("=" * 45)
print("Total entities:", total)
print("Verified entities:", verified)
print("Pending entities:", pending)
print(f"Completion: {verified / total * 100:.1f}%")

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
reports_path = "/kaggle/working/combined_medical_reports_clean.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")
reports_df = pd.read_csv(reports_path, dtype=str).fillna("")

pending_ids = (
    review_df.loc[
        review_df["verification_status"] != "VERIFIED",
        "report_id"
    ]
    .drop_duplicates()
)

if pending_ids.empty:
    print("All proposed annotations have been reviewed.")
else:
    report_id = pending_ids.iloc[0]
    report = reports_df[reports_df["report_id"] == report_id]

    if report.empty:
        print("Original report not found:", report_id)
    else:
        print("Report ID:", report_id)
        print("Dataset:", report.iloc[0]["source_dataset"])
        print("\nORIGINAL REPORT:\n")
        print(report.iloc[0]["report_text"])

        print("\nPROPOSED ANNOTATIONS:\n")
        display(
            review_df[review_df["report_id"] == report_id][
                [
                    "entity_text",
                    "proposed_label",
                    "final_label",
                    "assertion_status",
                    "label_correct",
                    "verification_status",
                    "review_notes",
                    "assertion_notes",
                ]
            ]
        )

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
review_df = pd.read_csv(review_path, dtype=str).fillna("")

report_id = "MED_00907"

updates = {
    "Right eye": {
        "final_label": "ANATOMY",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Anatomical location explicitly identified in the operative report."
        ),
        "assertion_notes": (
            "The right eye is the anatomical location of the operation."
        ),
    },
    "Cornea": {
        "final_label": "ANATOMY",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Anatomical structure explicitly mentioned in the description "
            "of the operation."
        ),
        "assertion_notes": (
            "The report documents incisions made through the cornea."
        ),
    },
    "Anterior chamber": {
        "final_label": "ANATOMY",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Anatomical structure explicitly mentioned in the operative report."
        ),
        "assertion_notes": (
            "The anterior chamber is explicitly mentioned in the procedure."
        ),
    },
    "Capsulorrhexis": {
        "final_label": "PROCEDURE",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "The report explicitly states that capsulorrhexis was performed."
        ),
        "assertion_notes": (
            "The capsulorrhexis procedure was performed during this operation."
        ),
    },
}

# Validate the rows before changing anything.
for entity_text in updates:
    matches = review_df[
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one row for {entity_text!r}; "
            f"found {len(matches)}."
        )

# Update the annotations.
for entity_text, values in updates.items():
    mask = (
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    )

    for column, value in values.items():
        review_df.loc[mask, column] = value

    review_df.loc[mask, "verification_status"] = "VERIFIED"

review_df.to_csv(review_path, index=False)

print("MED_00907 annotations saved successfully.")

display(
    review_df[review_df["report_id"] == report_id][
        [
            "entity_text",
            "final_label",
            "assertion_status",
            "label_correct",
            "verification_status",
        ]
    ]
)

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
review_df = pd.read_csv(review_path, dtype=str).fillna("")

total = len(review_df)
verified = (review_df["verification_status"] == "VERIFIED").sum()
pending = total - verified

print("=" * 45)
print("ANNOTATION REVIEW PROGRESS")
print("=" * 45)
print(f"Total entities: {total}")
print(f"Verified entities: {verified}")
print(f"Pending entities: {pending}")
print(f"Completion: {verified / total * 100:.1f}%")

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
reports_path = "/kaggle/working/combined_medical_reports_clean.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")
reports_df = pd.read_csv(reports_path, dtype=str).fillna("")

pending_ids = (
    review_df.loc[
        review_df["verification_status"] != "VERIFIED",
        "report_id"
    ]
    .drop_duplicates()
)

if pending_ids.empty:
    print("All proposed annotations have been reviewed.")
else:
    report_id = pending_ids.iloc[0]
    report = reports_df[reports_df["report_id"] == report_id]

    if report.empty:
        print("Original report not found:", report_id)
    else:
        print("Report ID:", report_id)
        print("Dataset:", report.iloc[0]["source_dataset"])
        print("\nORIGINAL REPORT:\n")
        print(report.iloc[0]["report_text"])

        print("\nPROPOSED ANNOTATIONS:\n")
        display(
            review_df[review_df["report_id"] == report_id][
                [
                    "entity_text",
                    "proposed_label",
                    "final_label",
                    "assertion_status",
                    "label_correct",
                    "verification_status",
                    "review_notes",
                    "assertion_notes",
                ]
            ]
        )

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
review_df = pd.read_csv(review_path, dtype=str).fillna("")

report_id = "MED_02280"

updates = {
    "Cystic fibrosis": {
        "final_label": "DISEASE",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Explicitly documented as a diagnosis in the patient's "
            "history and the report's impression."
        ),
        "assertion_notes": (
            "The report documents cystic fibrosis as an existing diagnosis."
        ),
    },
    "Chronic cough": {
        "final_label": "SYMPTOM",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Explicitly described in the interval history as a chronic "
            "cough that has recently increased slightly."
        ),
        "assertion_notes": (
            "The report documents chronic cough and a recent increase "
            "in the symptom."
        ),
    },
}

# Validate before making changes.
for entity_text in updates:
    matches = review_df[
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one row for {entity_text!r}; "
            f"found {len(matches)}."
        )

# Apply the review decisions.
for entity_text, values in updates.items():
    mask = (
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    )

    for column, value in values.items():
        review_df.loc[mask, column] = value

    review_df.loc[mask, "verification_status"] = "VERIFIED"

review_df.to_csv(review_path, index=False)

print("MED_02280 annotations saved successfully.")

display(
    review_df[review_df["report_id"] == report_id][
        [
            "entity_text",
            "final_label",
            "assertion_status",
            "label_correct",
            "verification_status",
        ]
    ]
)

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
review_df = pd.read_csv(review_path, dtype=str).fillna("")

total = len(review_df)
verified = (review_df["verification_status"] == "VERIFIED").sum()
pending = total - verified

print("=" * 45)
print("ANNOTATION REVIEW PROGRESS")
print("=" * 45)
print(f"Total entities: {total}")
print(f"Verified entities: {verified}")
print(f"Pending entities: {pending}")
print(f"Completion: {verified / total * 100:.1f}%")

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
reports_path = "/kaggle/working/combined_medical_reports_clean.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")
reports_df = pd.read_csv(reports_path, dtype=str).fillna("")

pending_ids = (
    review_df.loc[
        review_df["verification_status"] != "VERIFIED",
        "report_id"
    ]
    .drop_duplicates()
)

if pending_ids.empty:
    print("All proposed annotations have been reviewed.")
else:
    report_id = pending_ids.iloc[0]
    report = reports_df[reports_df["report_id"] == report_id]

    if report.empty:
        print("Original report not found:", report_id)
    else:
        print("Report ID:", report_id)
        print("Dataset:", report.iloc[0]["source_dataset"])
        print("\nORIGINAL REPORT:\n")
        print(report.iloc[0]["report_text"])

        print("\nPROPOSED ANNOTATIONS:\n")
        display(
            review_df[review_df["report_id"] == report_id][
                [
                    "entity_text",
                    "proposed_label",
                    "final_label",
                    "assertion_status",
                    "label_correct",
                    "verification_status",
                    "review_notes",
                    "assertion_notes",
                ]
            ]
        )

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
review_df = pd.read_csv(review_path, dtype=str).fillna("")

report_id = "MED_03739"

updates = {
    "Mediastinal contour": {
        "final_label": "ANATOMY",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Anatomical structure explicitly mentioned in the radiology "
            "report and described as within normal limits."
        ),
        "assertion_notes": (
            "The mediastinal contour is documented as within normal limits; "
            "no abnormal contour is reported."
        ),
    },
    "Lungs": {
        "final_label": "ANATOMY",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Anatomical structures explicitly mentioned and described "
            "as well-expanded and clear."
        ),
        "assertion_notes": (
            "The lungs are documented as well-expanded and clear."
        ),
    },
}

# Validate all target rows before editing.
for entity_text in updates:
    matches = review_df[
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one row for {entity_text!r}; "
            f"found {len(matches)}."
        )

# Apply the reviewed annotations.
for entity_text, values in updates.items():
    mask = (
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    )

    for column, value in values.items():
        review_df.loc[mask, column] = value

    review_df.loc[mask, "verification_status"] = "VERIFIED"

review_df.to_csv(review_path, index=False)

print("MED_03739 annotations saved successfully.")

display(
    review_df[review_df["report_id"] == report_id][
        [
            "entity_text",
            "final_label",
            "assertion_status",
            "label_correct",
            "verification_status",
        ]
    ]
)

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
review_df = pd.read_csv(review_path, dtype=str).fillna("")

total = len(review_df)
verified = (review_df["verification_status"] == "VERIFIED").sum()
pending = total - verified

print("=" * 45)
print("ANNOTATION REVIEW PROGRESS")
print("=" * 45)
print(f"Total entities: {total}")
print(f"Verified entities: {verified}")
print(f"Pending entities: {pending}")
print(f"Completion: {verified / total * 100:.1f}%")

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
reports_path = "/kaggle/working/combined_medical_reports_clean.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")
reports_df = pd.read_csv(reports_path, dtype=str).fillna("")

pending_ids = (
    review_df.loc[
        review_df["verification_status"] != "VERIFIED",
        "report_id"
    ]
    .drop_duplicates()
)

if pending_ids.empty:
    print("All proposed annotations have been reviewed.")
else:
    report_id = pending_ids.iloc[0]
    report = reports_df[reports_df["report_id"] == report_id]

    if report.empty:
        print("Original report not found:", report_id)
    else:
        print("Report ID:", report_id)
        print("Dataset:", report.iloc[0]["source_dataset"])
        print("\nORIGINAL REPORT:\n")
        print(report.iloc[0]["report_text"])

        print("\nPROPOSED ANNOTATIONS:\n")
        display(
            review_df[review_df["report_id"] == report_id][
                [
                    "entity_text",
                    "proposed_label",
                    "final_label",
                    "assertion_status",
                    "label_correct",
                    "verification_status",
                    "review_notes",
                    "assertion_notes",
                ]
            ]
        )

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
review_df = pd.read_csv(review_path, dtype=str).fillna("")

report_id = "MED_04039"

updates = {
    "Left arm pain": {
        "final_label": "SYMPTOM",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Explicitly stated as an indication for the radiological "
            "examination."
        ),
        "assertion_notes": (
            "The report documents left arm pain as an examination indication."
        ),
    },
    "Nausea": {
        "final_label": "SYMPTOM",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Explicitly stated as an indication for the radiological "
            "examination."
        ),
        "assertion_notes": (
            "The report documents nausea as an examination indication."
        ),
    },
    "Cardiomegaly": {
        "final_label": "RADIOLOGICAL_FINDING",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Enlarged heart is documented in the findings and cardiomegaly "
            "is explicitly stated in the impression."
        ),
        "assertion_notes": (
            "The radiology report documents cardiomegaly as present."
        ),
    },
}

# Validate every target row before making any changes.
for entity_text in updates:
    matches = review_df[
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one row for {entity_text!r}; "
            f"found {len(matches)}."
        )

# Apply the reviewed decisions.
for entity_text, values in updates.items():
    mask = (
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    )

    for column, value in values.items():
        review_df.loc[mask, column] = value

    review_df.loc[mask, "verification_status"] = "VERIFIED"

review_df.to_csv(review_path, index=False)

print("MED_04039 annotations saved successfully.")

display(
    review_df[review_df["report_id"] == report_id][
        [
            "entity_text",
            "final_label",
            "assertion_status",
            "label_correct",
            "verification_status",
        ]
    ]
)

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
review_df = pd.read_csv(review_path, dtype=str).fillna("")

total = len(review_df)
verified = (review_df["verification_status"] == "VERIFIED").sum()
pending = total - verified

print("=" * 45)
print("ANNOTATION REVIEW PROGRESS")
print("=" * 45)
print(f"Total entities: {total}")
print(f"Verified entities: {verified}")
print(f"Pending entities: {pending}")
print(f"Completion: {verified / total * 100:.1f}%")

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
reports_path = "/kaggle/working/combined_medical_reports_clean.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")
reports_df = pd.read_csv(reports_path, dtype=str).fillna("")

pending_ids = (
    review_df.loc[
        review_df["verification_status"] != "VERIFIED",
        "report_id"
    ]
    .drop_duplicates()
)

if pending_ids.empty:
    print("All proposed annotations have been reviewed.")
else:
    report_id = pending_ids.iloc[0]
    report = reports_df[reports_df["report_id"] == report_id]

    if report.empty:
        print("Original report not found:", report_id)
    else:
        print("Report ID:", report_id)
        print("Dataset:", report.iloc[0]["source_dataset"])
        print("\nORIGINAL REPORT:\n")
        print(report.iloc[0]["report_text"])

        print("\nPROPOSED ANNOTATIONS:\n")
        display(
            review_df[review_df["report_id"] == report_id][
                [
                    "entity_text",
                    "proposed_label",
                    "final_label",
                    "assertion_status",
                    "label_correct",
                    "verification_status",
                    "review_notes",
                    "assertion_notes",
                ]
            ]
        )

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
review_df = pd.read_csv(review_path, dtype=str).fillna("")

report_id = "MED_04978"

updates = {
    "Chest pain": {
        "final_label": "SYMPTOM",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Explicitly stated as an indication for the radiological examination."
        ),
        "assertion_notes": (
            "The report documents chest pain as an examination indication."
        ),
    },
    "Shortness of breath": {
        "final_label": "SYMPTOM",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Explicitly stated as an indication for the radiological examination."
        ),
        "assertion_notes": (
            "The report documents shortness of breath as an examination indication."
        ),
    },
    "Left pleural effusion": {
        "final_label": "RADIOLOGICAL_FINDING",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Explicitly documented in the radiology impression. The text "
            "immediately before this finding is redacted as XXXX; no missing "
            "word is inferred."
        ),
        "assertion_notes": (
            "The visible impression documents a left pleural effusion, "
            "although preceding wording is redacted."
        ),
    },
    "Airspace disease": {
        "final_label": "RADIOLOGICAL_FINDING",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Left basilar airspace disease is explicitly documented "
            "in the radiology impression."
        ),
        "assertion_notes": (
            "The impression documents associated left basilar airspace disease."
        ),
    },
    "Cardiomegaly": {
        "final_label": "RADIOLOGICAL_FINDING",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Moderate cardiomegaly is explicitly documented in the impression."
        ),
        "assertion_notes": (
            "The report documents moderate cardiomegaly as present."
        ),
    },
}

# Validate all rows before editing.
for entity_text in updates:
    matches = review_df[
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one row for {entity_text!r}; "
            f"found {len(matches)}."
        )

# Apply the review decisions.
for entity_text, values in updates.items():
    mask = (
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    )

    for column, value in values.items():
        review_df.loc[mask, column] = value

    review_df.loc[mask, "verification_status"] = "VERIFIED"

review_df.to_csv(review_path, index=False)

print("MED_04978 annotations saved successfully.")

display(
    review_df[review_df["report_id"] == report_id][
        [
            "entity_text",
            "final_label",
            "assertion_status",
            "label_correct",
            "verification_status",
        ]
    ]
)

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
review_df = pd.read_csv(review_path, dtype=str).fillna("")

total = len(review_df)
verified = (review_df["verification_status"] == "VERIFIED").sum()
pending = total - verified

print("=" * 45)
print("ANNOTATION REVIEW PROGRESS")
print("=" * 45)
print(f"Total entities: {total}")
print(f"Verified entities: {verified}")
print(f"Pending entities: {pending}")
print(f"Completion: {verified / total * 100:.1f}%")

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
reports_path = "/kaggle/working/combined_medical_reports_clean.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")
reports_df = pd.read_csv(reports_path, dtype=str).fillna("")

pending_ids = (
    review_df.loc[
        review_df["verification_status"] != "VERIFIED",
        "report_id"
    ]
    .drop_duplicates()
)

if pending_ids.empty:
    print("All proposed annotations have been reviewed.")
else:
    report_id = pending_ids.iloc[0]
    report = reports_df[reports_df["report_id"] == report_id]

    if report.empty:
        print("Original report not found:", report_id)
    else:
        print("Report ID:", report_id)
        print("Dataset:", report.iloc[0]["source_dataset"])
        print("\nORIGINAL REPORT:\n")
        print(report.iloc[0]["report_text"])

        print("\nPROPOSED ANNOTATIONS:\n")
        display(
            review_df[review_df["report_id"] == report_id][
                [
                    "entity_text",
                    "proposed_label",
                    "final_label",
                    "assertion_status",
                    "label_correct",
                    "verification_status",
                    "review_notes",
                    "assertion_notes",
                ]
            ]
        )

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
review_df = pd.read_csv(review_path, dtype=str).fillna("")

report_id = "MED_03835"

updates = {
    "Shortness of breath": {
        "final_label": "SYMPTOM",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Explicitly stated as an indication for the radiological examination."
        ),
        "assertion_notes": (
            "The report documents shortness of breath as an examination indication."
        ),
    },
    "Wheezing": {
        "final_label": "SYMPTOM",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Explicitly stated as an indication for the radiological examination."
        ),
        "assertion_notes": (
            "The report documents wheezing as an examination indication."
        ),
    },
    "Lung": {
        "final_label": "ANATOMY",
        "label_correct": "YES",
        "assertion_status": "PRESENT",
        "review_notes": (
            "Anatomical structure explicitly mentioned as the location "
            "of the nodular opacity."
        ),
        "assertion_notes": (
            "The lung is documented as an anatomical location. The nearby "
            "opacity is described as possibly representing an infiltrate "
            "or nodule; that uncertainty does not apply to the anatomy entity."
        ),
    },
    "Pleural effusion": {
        "final_label": "RADIOLOGICAL_FINDING",
        "label_correct": "YES",
        "assertion_status": "ABSENT",
        "review_notes": (
            "The report explicitly states 'No pleural effusion.' "
            "The finding is negated."
        ),
        "assertion_notes": (
            "Explicitly absent according to the radiology report."
        ),
    },
}

# Validate all target rows before editing.
for entity_text in updates:
    matches = review_df[
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one row for {entity_text!r}; "
            f"found {len(matches)}."
        )

# Apply the reviewed decisions.
for entity_text, values in updates.items():
    mask = (
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    )

    for column, value in values.items():
        review_df.loc[mask, column] = value

    review_df.loc[mask, "verification_status"] = "VERIFIED"

review_df.to_csv(review_path, index=False)

print("MED_03835 annotations saved successfully.")

display(
    review_df[review_df["report_id"] == report_id][
        [
            "entity_text",
            "final_label",
            "assertion_status",
            "label_correct",
            "verification_status",
        ]
    ]
)

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
review_df = pd.read_csv(review_path, dtype=str).fillna("")

total = len(review_df)
verified = (review_df["verification_status"] == "VERIFIED").sum()
pending = total - verified

print("=" * 45)
print("ANNOTATION REVIEW PROGRESS")
print("=" * 45)
print(f"Total entities: {total}")
print(f"Verified entities: {verified}")
print(f"Pending entities: {pending}")
print(f"Completion: {verified / total * 100:.1f}%")

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
reports_path = "/kaggle/working/combined_medical_reports_clean.csv"

review_df = pd.read_csv(review_path, dtype=str).fillna("")
reports_df = pd.read_csv(reports_path, dtype=str).fillna("")

pending_ids = (
    review_df.loc[
        review_df["verification_status"] != "VERIFIED",
        "report_id"
    ]
    .drop_duplicates()
)

if pending_ids.empty:
    print("All proposed annotations have been reviewed.")
else:
    report_id = pending_ids.iloc[0]
    report = reports_df[reports_df["report_id"] == report_id]

    if report.empty:
        print("Original report not found:", report_id)
    else:
        print("Report ID:", report_id)
        print("Dataset:", report.iloc[0]["source_dataset"])
        print("\nORIGINAL REPORT:\n")
        print(report.iloc[0]["report_text"])

        print("\nPROPOSED ANNOTATIONS:\n")
        display(
            review_df[review_df["report_id"] == report_id][
                [
                    "entity_text",
                    "proposed_label",
                    "final_label",
                    "assertion_status",
                    "label_correct",
                    "verification_status",
                    "review_notes",
                    "assertion_notes",
                ]
            ]
        )

In [ ]:
import pandas as pd

review_path = "/kaggle/working/medical_ner_manual_review.csv"
review_df = pd.read_csv(review_path, dtype=str).fillna("")

report_id = "MED_02419"

updates = {
    "Pneumothorax": (
        "RADIOLOGICAL_FINDING",
        "The report explicitly states 'No pneumothorax'.",
    ),
    "Airspace consolidation": (
        "RADIOLOGICAL_FINDING",
        "The report explicitly states 'No airspace consolidation'.",
    ),
    "Pleural effusion": (
        "RADIOLOGICAL_FINDING",
        "The report explicitly states 'No pleural effusion'.",
    ),
}

# Validate all rows before updating.
for entity_text in updates:
    matches = review_df[
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one row for {entity_text!r}; "
            f"found {len(matches)}."
        )

# Save the reviewed annotations.
for entity_text, (label, note) in updates.items():
    mask = (
        (review_df["report_id"] == report_id)
        & (review_df["entity_text"] == entity_text)
    )

    review_df.loc[mask, "final_label"] = label
    review_df.loc[mask, "label_correct"] = "YES"
    review_df.loc[mask, "assertion_status"] = "ABSENT"
    review_df.loc[mask, "review_notes"] = note
    review_df.loc[mask, "assertion_notes"] = (
        "The radiology report explicitly negates this finding."
    )
    review_df.loc[mask, "verification_status"] = "VERIFIED"

review_df.to_csv(review_path, index=False)

print("MED_02419 saved successfully!")

display(
    review_df[review_df["report_id"] == report_id][
        [
            "entity_text",
            "final_label",
            "assertion_status",
            "verification_status",
        ]
    ]
)

verified = (review_df["verification_status"] == "VERIFIED").sum()

print(f"\nVerified entities: {verified}/{len(review_df)}")
print(f"Completion: {verified / len(review_df) * 100:.1f}%")

In [ ]:
import os
import json
import zipfile
import shutil
from datetime import datetime

BASE = "/kaggle/working"
BACKUP_DIR = os.path.join(BASE, "medical_report_analyzer_backup")

os.makedirs(BACKUP_DIR, exist_ok=True)

# Important project files to preserve.
important_files = [
    "mtsamples_clean.csv",
    "iu_xray_reports.csv",
    "iu_xray_reports_clean.csv",
    "combined_medical_reports_clean.csv",
    "medical_ner_annotation_template.csv",
    "medical_ner_annotation_progress.csv",
    "medical_ner_annotation_progress_backup.csv",
    "medical_ner_manual_review.csv",
    "medical_ner_proposed_annotations_v2.csv",
    "medical_ner_annotation_schema.json",
    "medical_ner_assertion_schema.json",
    "medical_ner_schema_validation.csv",
    "medical_ner_annotation_audit.json",
    "medical_ner_evaluation_results.csv",
    "medical_ner_prediction_details.csv",
]

# Copy existing project files.
copied_files = []

for filename in important_files:
    source = os.path.join(BASE, filename)

    if os.path.isfile(source):
        destination = os.path.join(BACKUP_DIR, filename)
        shutil.copy2(source, destination)
        copied_files.append(filename)

# Preserve the train/validation/test split files.
split_dir = os.path.join(BASE, "medical_report_splits")

if os.path.isdir(split_dir):
    destination_dir = os.path.join(BACKUP_DIR, "medical_report_splits")

    shutil.copytree(
        split_dir,
        destination_dir,
        dirs_exist_ok=True
    )

    for root, dirs, files in os.walk(destination_dir):
        for filename in files:
            relative_path = os.path.relpath(
                os.path.join(root, filename),
                BACKUP_DIR
            )
            copied_files.append(relative_path)

# Create a readable progress summary.
review_path = os.path.join(
    BASE,
    "medical_ner_manual_review.csv"
)

if os.path.isfile(review_path):
    import pandas as pd

    review_df = pd.read_csv(review_path, dtype=str).fillna("")

    total = len(review_df)
    verified = (
        review_df["verification_status"] == "VERIFIED"
    ).sum()

    progress = {
        "backup_created": datetime.now().isoformat(timespec="seconds"),
        "total_annotation_rows": int(total),
        "verified_rows": int(verified),
        "pending_rows": int(total - verified),
        "completion_percent": round(verified / total * 100, 1)
            if total else 0,
        "important_note": (
            "VERIFIED means accepted during project review; "
            "it does not imply independent clinical validation."
        ),
        "copied_files": copied_files,
    }

    with open(
        os.path.join(BACKUP_DIR, "project_progress.json"),
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(progress, f, indent=2)

# Create a timestamped ZIP archive.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

zip_path = os.path.join(
    BASE,
    f"medical_report_analyzer_backup_{timestamp}.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:

    for root, dirs, files in os.walk(BACKUP_DIR):
        for filename in files:
            full_path = os.path.join(root, filename)

            archive_name = os.path.relpath(
                full_path,
                BACKUP_DIR
            )

            zipf.write(full_path, archive_name)

print("=" * 60)
print("PROJECT BACKUP CREATED")
print("=" * 60)

print("\nFiles included:", len(copied_files))
print("ZIP location:", zip_path)
print("ZIP size:", round(os.path.getsize(zip_path) / (1024 * 1024), 2), "MB")

print("\nIncluded files:")
for filename in copied_files:
    print(" -", filename)

print("\nYour annotation progress has been included in the backup.")

In [ ]:
import os

print("Checking important project files...\n")

files_to_check = [
    "medical_ner_manual_review.csv",
    "combined_medical_reports_clean.csv",
    "medical_ner_annotation_schema.json",
    "medical_ner_assertion_schema.json",
    "medical_ner_proposed_annotations_v2.csv",
]

for filename in files_to_check:
    path = os.path.join("/kaggle/working", filename)

    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"✅ {filename} ({size_kb:.1f} KB)")
    else:
        print(f"❌ Missing: {filename}")

In [ ]:
import os
import zipfile
from datetime import datetime

BASE = "/kaggle/working"

# Timestamp the backup so older backups aren't overwritten.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = os.path.join(
    BASE,
    f"medical_report_analyzer_backup_{timestamp}.zip"
)

# Files to preserve.
important_files = [
    "mtsamples_clean.csv",
    "iu_xray_reports.csv",
    "iu_xray_reports_clean.csv",
    "combined_medical_reports_clean.csv",
    "medical_ner_annotation_template.csv",
    "medical_ner_annotation_progress.csv",
    "medical_ner_annotation_progress_backup.csv",
    "medical_ner_manual_review.csv",
    "medical_ner_proposed_annotations_v2.csv",
    "medical_ner_annotation_schema.json",
    "medical_ner_assertion_schema.json",
    "medical_ner_schema_validation.csv",
    "medical_ner_annotation_audit.json",
    "medical_ner_evaluation_results.csv",
    "medical_ner_prediction_details.csv",
]

included_files = []

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as backup:

    # Add the existing project files.
    for filename in important_files:
        path = os.path.join(BASE, filename)

        if os.path.isfile(path):
            backup.write(path, arcname=filename)
            included_files.append(filename)

    # Include the dataset splits.
    split_dir = os.path.join(BASE, "medical_report_splits")

    if os.path.isdir(split_dir):
        for root, dirs, files in os.walk(split_dir):
            for filename in files:
                path = os.path.join(root, filename)
                arcname = os.path.relpath(path, BASE)
                backup.write(path, arcname)
                included_files.append(arcname)

print("=" * 55)
print("BACKUP CREATED SUCCESSFULLY")
print("=" * 55)
print("Location:", zip_path)
print("Files included:", len(included_files))
print("Size:", round(os.path.getsize(zip_path) / (1024 * 1024), 2), "MB")

print("\nBackup contents:")
for filename in included_files:
    print("✅", filename)